# LeetCode 327 — Count of Range Sum (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countRangeSum(self, nums: List[int], lower: int, upper: int) -> int:

```

Đề bài nghe rất "đời thường": **đếm xem có bao nhiêu đoạn con liên tiếp có tổng nằm trong một khoảng cho trước**. Người mới nào cũng nghĩ ra ngay cách thử mọi đoạn — và cách đó **đúng**. Cái khó (vì sao bài là Hard) nằm ở chỗ: với `n = 10^5` phần tử có tới khoảng `5·10^9` đoạn, thử từng đoạn là quá chậm; còn mảng lại có **số âm** nên mẹo "cửa sổ trượt" quen thuộc cũng không dùng được.

Lời giải tối ưu là sự kết hợp của **hai ý tưởng nền tảng** mà notebook sẽ dạy từ đầu:

1. **Prefix sum (tổng tiền tố)** — biến "tổng của một đoạn" thành "hiệu của hai con số".
2. Một kỹ thuật **đếm nhanh "có bao nhiêu số đã gặp nằm trong một khoảng"** — ta sẽ học 3 phiên bản: list sắp xếp + `bisect`, **merge sort đếm khi trộn**, và **Fenwick Tree + nén tọa độ**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề, **liệt kê bằng tay** mọi đoạn con và tổng của chúng trên ví dụ nhỏ.
2. **Cách 1 — Brute force**: bản `O(n³)` (cộng lại từ đầu) rồi bản `O(n²)` (tổng chạy — running sum).
3. **Prefix sum** từ căn bản: công thức vàng `S(i, j) = P[j+1] − P[i]`; **Cách 2** — biến bài toán thành **đếm cặp chỉ số** `a < b` với `lower ≤ P[b] − P[a] ≤ upper`.
4. **Chuyển vế bất đẳng thức** để có "cửa sổ" `[P[b] − upper, P[b] − lower]` — bước then chốt của mọi lời giải nhanh.
5. **Cách 3 — List sắp xếp + `bisect`**: học tìm kiếm nhị phân (binary search) từ đầu; hiểu vì sao `insort` vẫn tốn `O(n)`.
6. **Cách 4 — Merge sort đếm khi trộn** (cách chính, `O(n log n)`): đệ quy chia đôi, trộn hai mảng đã sắp xếp, two pointers đếm cặp chéo; trace từng nút đệ quy.
7. **Cách 5 — Fenwick Tree (BIT) + nén tọa độ**: dạy BIT từ con số `i & -i`.
8. Code nộp LeetCode, test đối chiếu chéo cả 5 cách, vì sao đúng, so sánh, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho một mảng số nguyên `nums` và hai số nguyên `lower`, `upper`. Hãy trả về **số lượng range sum** nằm trong đoạn `[lower, upper]` (tính cả hai đầu).

**Range sum** `S(i, j)` được định nghĩa là tổng các phần tử của `nums` từ chỉ số `i` tới chỉ số `j` (tính cả hai đầu), với `i <= j`.

**Example 1**

```text
Input:  nums = [-2,5,-1], lower = -2, upper = 2
Output: 3
Explanation: Ba đoạn thỏa mãn là [0,0], [2,2] và [0,2], với tổng lần lượt là -2, -1, 2.
```

**Example 2**

```text
Input:  nums = [0], lower = 0, upper = 0
Output: 1
```

**Constraints**

- `1 <= nums.length <= 10^5`
- `-2^31 <= nums[i] <= 2^31 - 1`
- `-10^5 <= lower <= upper <= 10^5`
- Đáp án được **đảm bảo** vừa với số nguyên **32-bit**.

Ba chi tiết cần "khắc cốt ghi tâm" ngay từ đầu:

- "Đoạn" ở đây là **đoạn con liên tiếp** (subarray) `nums[i], nums[i+1], ..., nums[j]` — không phải chọn phần tử tùy ý.
- Vì `i <= j`, mỗi đoạn có **ít nhất 1 phần tử** (không tính đoạn rỗng).
- `nums[i]` có thể **âm** và rất lớn (tới khoảng ±2,1 tỉ) → tổng một đoạn có thể tới khoảng `10^5 · 2^31 ≈ 2·10^14`. Python tự xử lý số nguyên lớn, nhưng nếu viết C++/Java phải dùng `long long`/`long`.

## 2. Hiểu đề bằng tay: liệt kê mọi đoạn của Example 1

`nums = [-2, 5, -1]`, `lower = -2`, `upper = 2`. Mảng có 3 phần tử nên có `3 + 2 + 1 = 6` đoạn:

| (i, j) | Đoạn `nums[i..j]` | Tổng | Trong `[-2, 2]`? |
|:---:|:---|---:|:---:|
| (0, 0) | `[-2]` | -2 | ✅ |
| (0, 1) | `[-2, 5]` | 3 | ❌ |
| (0, 2) | `[-2, 5, -1]` | 2 | ✅ |
| (1, 1) | `[5]` | 5 | ❌ |
| (1, 2) | `[5, -1]` | 4 | ❌ |
| (2, 2) | `[-1]` | -1 | ✅ |

→ Có **3** đoạn hợp lệ, khớp `Output: 3`. Chú ý: biên `-2` và `2` **được tính** (đoạn `(0,0)` có tổng đúng bằng `lower`, đoạn `(0,2)` có tổng đúng bằng `upper`).

**Python cần biết cho cell sau:**

- `nums[i:j + 1]` là **slicing** (cắt list): lấy các phần tử từ chỉ số `i` tới **trước** `j + 1`, tức là tới `j`. Đầu cuối của slicing **không được lấy**, vì vậy phải viết `j + 1`.
- `lower <= s <= upper` là **so sánh nối chuỗi** của Python, nghĩa là `lower <= s and s <= upper`.
- `sum(điều_kiện for ...)` đếm số lần điều kiện đúng, vì `True` được cộng như `1`, `False` như `0`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

examples = [
    ([-2, 5, -1], -2, 2, 3),   # (nums, lower, upper, expected)
    ([0], 0, 0, 1),
]

def list_all_range_sums(nums):
    """Liệt kê mọi đoạn (i, j) với i <= j cùng với tổng của nó."""
    result = []
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            segment = nums[i:j + 1]          # j + 1 vì đầu cuối của slicing không được lấy
            result.append(((i, j), segment, sum(segment)))
    return result

nums, lower, upper, expected = examples[0]
all_sums = list_all_range_sums(nums)
for (i, j), segment, s in all_sums:
    mark = "✅" if lower <= s <= upper else "❌"
    print(f"(i={i}, j={j})  đoạn={str(segment):<12} tổng={s:>2}  {mark}")
print("Số đoạn hợp lệ:", sum(lower <= s <= upper for _, _, s in all_sums), "| expected =", expected)

(i=0, j=0)  đoạn=[-2]         tổng=-2  ✅
(i=0, j=1)  đoạn=[-2, 5]      tổng= 3  ❌
(i=0, j=2)  đoạn=[-2, 5, -1]  tổng= 2  ✅
(i=1, j=1)  đoạn=[5]          tổng= 5  ❌
(i=1, j=2)  đoạn=[5, -1]      tổng= 4  ❌
(i=2, j=2)  đoạn=[-1]         tổng=-1  ✅
Số đoạn hợp lệ: 3 | expected = 3


### Hai quan sát quan trọng trước khi giải

**Quan sát 1 — Có rất nhiều đoạn.** Với mảng dài `n`: đoạn bắt đầu tại `0` có `n` cách chọn điểm kết thúc, bắt đầu tại `1` có `n − 1` cách, ..., bắt đầu tại `n − 1` có `1` cách. Tổng cộng `n + (n−1) + ... + 1 = n(n+1)/2` đoạn. Với `n = 10^5` là khoảng **5 tỉ** đoạn — liệt kê từng đoạn chắc chắn quá chậm. Ta cần cách **đếm cả nhóm đoạn một lúc** mà không cần "chạm" vào từng đoạn.

**Quan sát 2 — Số âm phá hỏng "cửa sổ trượt".** **Sliding window** (cửa sổ trượt) là kỹ thuật dùng hai con trỏ `left`, `right`: kéo `right` để mở rộng, kéo `left` để thu hẹp. Nó chỉ hoạt động khi "mở rộng đoạn thì tổng chỉ tăng" — điều này đúng khi mọi phần tử **không âm**. Ở bài này, kéo dài đoạn bắt đầu tại `0` của Example 1 cho tổng `-2 → 3 → 2`: **tăng rồi lại giảm**. Không có tính đơn điệu (monotonic) thì ta không biết nên kéo con trỏ nào → phải tìm hướng khác. (Phần "Lỗi hay gặp" sẽ cho thấy sliding window trả sai thật.)

In [2]:
for n in [3, 10, 1000, 10**5]:
    print(f"n = {n:>6}: số đoạn con = n(n+1)/2 = {n * (n + 1) // 2:,}")

print()
nums = [-2, 5, -1]
running = 0
for j in range(len(nums)):
    running += nums[j]
    print(f"Kéo dài đoạn bắt đầu tại 0 tới j = {j}: tổng = {running}")
print("-> Tổng tăng rồi giảm: không đơn điệu, sliding window không áp dụng được.")

n =      3: số đoạn con = n(n+1)/2 = 6
n =     10: số đoạn con = n(n+1)/2 = 55
n =   1000: số đoạn con = n(n+1)/2 = 500,500
n = 100000: số đoạn con = n(n+1)/2 = 5,000,050,000

Kéo dài đoạn bắt đầu tại 0 tới j = 0: tổng = -2
Kéo dài đoạn bắt đầu tại 0 tới j = 1: tổng = 3
Kéo dài đoạn bắt đầu tại 0 tới j = 2: tổng = 2
-> Tổng tăng rồi giảm: không đơn điệu, sliding window không áp dụng được.


## 3. Cách 1a — Brute force 3 vòng lặp: thử mọi đoạn, cộng lại từ đầu

Cách người mới nghĩ ra đầu tiên, dịch thẳng từ đề:

- Vòng 1: chọn điểm bắt đầu `i` từ `0` tới `n − 1`.
- Vòng 2: chọn điểm kết thúc `j` từ `i` tới `n − 1` (vì đề yêu cầu `i <= j`).
- Vòng 3: cộng `nums[i] + ... + nums[j]` bằng một vòng `for k in range(i, j + 1)`.
- Nếu tổng nằm trong `[lower, upper]` thì tăng biến đếm.

Code này **chắc chắn đúng** vì nó làm y hệt định nghĩa. Ta sẽ giữ nó làm **"trọng tài"** để kiểm tra các cách nhanh hơn ở cuối notebook.

Độ phức tạp: có `~n²/2` đoạn, mỗi đoạn cộng tối đa `n` số → **`O(n³)`** thời gian, `O(1)` bộ nhớ.

In [3]:
def count_range_sum_cubic(nums: List[int], lower: int, upper: int) -> int:
    n = len(nums)
    count = 0
    for i in range(n):                  # điểm bắt đầu
        for j in range(i, n):           # điểm kết thúc, j >= i
            total = 0
            for k in range(i, j + 1):   # cộng lại TỪ ĐẦU cho mỗi đoạn -> lãng phí
                total += nums[k]
            if lower <= total <= upper:
                count += 1
    return count

for nums, lower, upper, expected in examples:
    print(nums, lower, upper, "->", count_range_sum_cubic(nums, lower, upper), "| expected:", expected)

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


## 4. Cách 1b — Brute force `O(n²)` với tổng chạy (running sum)

**Cách 1a lãng phí ở đâu?** Nhìn các đoạn cùng điểm bắt đầu `i = 0` của Example 1:

```text
S(0,0) = -2
S(0,1) = -2 + 5        = S(0,0) + nums[1] = -2 + 5  = 3
S(0,2) = -2 + 5 + (-1) = S(0,1) + nums[2] =  3 + (-1) = 2
```

Tổng của đoạn dài hơn **chứa sẵn** tổng của đoạn ngắn hơn. Vậy khi `i` cố định và `j` tăng dần, ta **chỉ cần cộng thêm `nums[j]`** vào tổng cũ, không cần cộng lại từ đầu. Biến lưu "tổng tới hiện tại" gọi là **running sum** (tổng chạy).

Đừng quên: khi đổi sang điểm bắt đầu `i` mới, phải **reset** `total = 0`.

Vòng `k` biến mất → **`O(n²)`** thời gian, `O(1)` bộ nhớ.

In [4]:
def count_range_sum_brute(nums: List[int], lower: int, upper: int) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        total = 0                       # reset khi đổi điểm bắt đầu
        for j in range(i, n):
            total += nums[j]            # total bây giờ = S(i, j)
            if lower <= total <= upper:
                count += 1
    return count

for nums, lower, upper, expected in examples:
    print(nums, lower, upper, "->", count_range_sum_brute(nums, lower, upper), "| expected:", expected)

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


### Trace cách 1b trên Example 1

| i | j | `nums[j]` | `total` sau khi cộng | Trong `[-2, 2]`? | `count` |
|:---:|:---:|---:|---:|:---:|:---:|
| 0 | 0 | -2 | -2 | ✅ | 1 |
| 0 | 1 | 5 | 3 | ❌ | 1 |
| 0 | 2 | -1 | 2 | ✅ | 2 |
| 1 | 1 | 5 | 5 | ❌ | 2 |
| 1 | 2 | -1 | 4 | ❌ | 2 |
| 2 | 2 | -1 | -1 | ✅ | 3 |

Để ý dòng `i = 1, j = 1`: `total` bắt đầu lại từ `0` rồi cộng `5`, chứ không phải `2 + 5`. Cell dưới in đúng bảng này.

In [5]:
def trace_brute(nums, lower, upper):
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            ok = lower <= total <= upper
            if ok:
                count += 1
            print(f"i={i}, j={j}, nums[j]={nums[j]:>2}, total={total:>2}, "
                  f"trong [{lower}, {upper}]? {'✅' if ok else '❌'}  count={count}")
    return count

print("Kết quả:", trace_brute([-2, 5, -1], -2, 2))

i=0, j=0, nums[j]=-2, total=-2, trong [-2, 2]? ✅  count=1
i=0, j=1, nums[j]= 5, total= 3, trong [-2, 2]? ❌  count=1
i=0, j=2, nums[j]=-1, total= 2, trong [-2, 2]? ✅  count=2
i=1, j=1, nums[j]= 5, total= 5, trong [-2, 2]? ❌  count=2
i=1, j=2, nums[j]=-1, total= 4, trong [-2, 2]? ❌  count=2
i=2, j=2, nums[j]=-1, total=-1, trong [-2, 2]? ✅  count=3
Kết quả: 3


### Cách 1 vướng ở đâu?

`O(n²)` với `n = 10^5` là khoảng `5·10^9` lần cộng + so sánh. Python thuần chạy cỡ vài chục triệu phép đơn giản mỗi giây → cần **hàng trăm giây**, trong khi LeetCode chỉ cho cỡ vài giây → **Time Limit Exceeded (TLE)**.

Cell dưới đo thử thời gian với `n = 2000` rồi nhân lên: `n` gấp 50 lần thì `n²` gấp `50² = 2500` lần (con số in ra phụ thuộc máy, nhưng chắc chắn quá chậm).

**Nút thắt:** ta đang xử lý **từng đoạn một**. Muốn nhanh hơn, phải có cách **đếm một lúc cả nhóm đoạn** cùng điểm kết thúc. Bước đầu tiên để làm được điều đó là biểu diễn tổng đoạn bằng **hiệu của hai con số** — đó là prefix sum.

In [6]:
import random
import time

random.seed(42)
n = 2000
nums = [random.randint(-100, 100) for _ in range(n)]
start = time.perf_counter()
ans = count_range_sum_brute(nums, -50, 50)
elapsed = time.perf_counter() - start
print(f"n = {n}: đáp án = {ans}, mất khoảng {elapsed:.2f} giây")
print(f"Ước lượng cho n = 100000 (gấp 2500 lần): khoảng {elapsed * 2500:.0f} giây -> TLE")

n = 2000: đáp án = 55903, mất khoảng 0.12 giây
Ước lượng cho n = 100000 (gấp 2500 lần): khoảng 312 giây -> TLE


## 5. Kiến thức nền 1 — Prefix sum (tổng tiền tố)

**Prefix** nghĩa là "phần đầu". **Prefix sum** `P[k]` là tổng của `k` phần tử **đầu tiên** của mảng:

```text
P[0] = 0                                (tổng của 0 phần tử: chưa cộng gì)
P[1] = nums[0]
P[2] = nums[0] + nums[1]
...
P[k] = nums[0] + nums[1] + ... + nums[k-1]
```

Với `nums = [-2, 5, -1]`:

| k | 0 | 1 | 2 | 3 |
|---|:---:|:---:|:---:|:---:|
| Ý nghĩa | (chưa lấy gì) | `-2` | `-2 + 5` | `-2 + 5 + (-1)` |
| `P[k]` | **0** | **-2** | **3** | **2** |

Quy tắc xây dựng: `P[k + 1] = P[k] + nums[k]` — mỗi bước chỉ cộng thêm một số, nên xây cả mảng `P` mất `O(n)`.

**Hai điều người mới hay quên:**

- `P` có **`n + 1` phần tử** (nhiều hơn `nums` một ô).
- `P[0] = 0` **không phải thừa**: nó là "tổng của đoạn rỗng phía trước index 0" và giúp các đoạn **bắt đầu từ index 0** cũng viết được bằng hiệu (xem công thức ngay sau).

Trong Python, `itertools.accumulate(nums, initial=0)` tạo đúng dãy này (tham số `initial` có từ Python 3.8).

In [7]:
from itertools import accumulate

def build_prefix(nums):
    P = [0]                    # P[0] = 0
    for x in nums:
        P.append(P[-1] + x)    # P[k+1] = P[k] + nums[k]   (P[-1] là phần tử cuối hiện tại)
    return P

nums = [-2, 5, -1]
print("build_prefix :", build_prefix(nums))
print("accumulate   :", list(accumulate(nums, initial=0)))
print("len(nums) =", len(nums), "| len(P) =", len(build_prefix(nums)), "(luôn nhiều hơn 1)")

build_prefix : [0, -2, 3, 2]
accumulate   : [0, -2, 3, 2]
len(nums) = 3 | len(P) = 4 (luôn nhiều hơn 1)


### Công thức vàng: `S(i, j) = P[j + 1] − P[i]`

Viết hai prefix sum chồng lên nhau:

```text
P[j+1] = nums[0] + ... + nums[i-1] + nums[i] + ... + nums[j]
P[i]   = nums[0] + ... + nums[i-1]
-----------------------------------------------------------------
hiệu   =                             nums[i] + ... + nums[j]   = S(i, j)
```

Phần đầu `nums[0] + ... + nums[i-1]` **triệt tiêu**, chỉ còn đúng đoạn `i..j`. Thử với Example 1 (`P = [0, -2, 3, 2]`):

- `S(0, 2) = P[3] − P[0] = 2 − 0 = 2` ✅ (đúng bằng `-2 + 5 - 1`)
- `S(1, 2) = P[3] − P[1] = 2 − (−2) = 4` ✅ (đúng bằng `5 - 1`)
- `S(2, 2) = P[3] − P[2] = 2 − 3 = −1` ✅
- `S(0, 0) = P[1] − P[0] = −2 − 0 = −2` ✅ ← nhờ có `P[0] = 0`

Sau khi xây `P` một lần (`O(n)`), **mỗi tổng đoạn tính trong `O(1)`**. Cell dưới kiểm tra công thức trên mọi đoạn của một mảng ngẫu nhiên.

In [8]:
import random

random.seed(0)
nums = [random.randint(-9, 9) for _ in range(6)]
P = build_prefix(nums)
print("nums =", nums)
print("P    =", P)

checked = 0
for i in range(len(nums)):
    for j in range(i, len(nums)):
        assert sum(nums[i:j + 1]) == P[j + 1] - P[i]
        checked += 1
print(f"Đã kiểm tra {checked} đoạn: S(i, j) == P[j+1] - P[i] ✅")
print(f"Ví dụ: S(1, 3) = sum({nums[1:4]}) = {sum(nums[1:4])};  P[4] - P[1] = ({P[4]}) - ({P[1]}) = {P[4] - P[1]}")

nums = [3, 4, -8, -1, 7, 6]
P    = [0, 3, 7, -1, -2, 5, 11]
Đã kiểm tra 21 đoạn: S(i, j) == P[j+1] - P[i] ✅
Ví dụ: S(1, 3) = sum([4, -8, -1]) = -5;  P[4] - P[1] = (-2) - (3) = -5


## 6. Cách 2 — Biến bài toán "đoạn con" thành bài toán "đếm cặp"

Đặt `a = i` và `b = j + 1`. Vì `0 ≤ i ≤ j ≤ n − 1` nên `0 ≤ a < b ≤ n`. Mỗi đoạn `(i, j)` ứng với **đúng một** cặp `(a, b)` với `a < b`, và ngược lại (cặp `(a, b)` ứng với đoạn `(a, b − 1)`). Công thức vàng thành `S = P[b] − P[a]`.

> **Bài toán mới:** trên mảng `P` (dài `n + 1`), đếm số cặp chỉ số `(a, b)` với **`a < b`** sao cho **`lower ≤ P[b] − P[a] ≤ upper`**.

Liệt kê cho Example 1, `P = [0, -2, 3, 2]`:

| (a, b) | Đoạn `(i, j) = (a, b−1)` | `P[b] − P[a]` | Hợp lệ? |
|:---:|:---:|:---|:---:|
| (0, 1) | (0, 0) | `-2 − 0 = -2` | ✅ |
| (0, 2) | (0, 1) | `3 − 0 = 3` | ❌ |
| (0, 3) | (0, 2) | `2 − 0 = 2` | ✅ |
| (1, 2) | (1, 1) | `3 − (−2) = 5` | ❌ |
| (1, 3) | (1, 2) | `2 − (−2) = 4` | ❌ |
| (2, 3) | (2, 2) | `2 − 3 = −1` | ✅ |

Hai lưu ý:

- **`a < b` ngặt**: cặp `a = b` ứng với "đoạn rỗng" (tổng `0`) — đề **không** tính.
- **Thứ tự quan trọng**: luôn là `P[b] − P[a]` với `b` ở **bên phải**. Đổi thành `P[a] − P[b]` là đếm sai.

Nếu duyệt mọi cặp thì vẫn `O(n²)` — **chưa nhanh hơn**. Nhưng bài toán giờ có dạng "**đếm cặp `a < b` thỏa điều kiện trên giá trị**", họ hàng với Two Sum và bài kinh điển "đếm nghịch thế". Đây là bàn đạp cho mọi cách nhanh phía sau.

In [9]:
def count_range_sum_prefix_pairs(nums: List[int], lower: int, upper: int) -> int:
    P = build_prefix(nums)
    count = 0
    for b in range(1, len(P)):          # b: đầu phải (đoạn kết thúc tại b - 1)
        for a in range(b):              # a < b: đầu trái
            if lower <= P[b] - P[a] <= upper:
                count += 1
    return count

for nums, lower, upper, expected in examples:
    got = count_range_sum_prefix_pairs(nums, lower, upper)
    print(nums, lower, upper, "->", got, "| expected:", expected)
    assert got == expected

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


### Bước then chốt: cố định `b`, chuyển vế ra một "cửa sổ" cho `P[a]`

Cố định đầu phải `b`. Điều kiện `lower ≤ P[b] − P[a] ≤ upper` gồm hai bất đẳng thức, chuyển vế từng cái:

- `P[b] − P[a] ≥ lower`  ⟺  `P[a] ≤ P[b] − lower`
- `P[b] − P[a] ≤ upper`  ⟺  `P[a] ≥ P[b] − upper`

Gộp lại:

> **`P[b] − upper  ≤  P[a]  ≤  P[b] − lower`**

⚠️ Để ý **chéo dấu**: `upper` đi với **cận dưới**, `lower` đi với **cận trên** (trừ một số lớn hơn thì được số nhỏ hơn). Đây là chỗ sai nhiều nhất của bài.

Ví dụ số: `b = 3`, `P[3] = 2`, `lower = −2`, `upper = 2` → cửa sổ `[2 − 2, 2 − (−2)] = [0, 4]`. Các `P[a]` đứng trước: `P[0] = 0` ✅, `P[1] = −2` ❌, `P[2] = 3` ✅ → **2** cặp, đúng là `(0, 3)` và `(2, 3)` trong bảng trên.

Vậy với mỗi `b` (duyệt từ trái sang phải), câu hỏi trở thành:

> "Trong các số `P[0], P[1], ..., P[b−1]` **đã gặp**, có **bao nhiêu** số nằm trong cửa sổ `[P[b] − upper, P[b] − lower]`?"

Rất giống Two Sum ("duyệt sang phải, hỏi về những số đã gặp ở bên trái"). Nhưng Two Sum hỏi "có số nào **bằng** `needed` không" → `dict` trả lời `O(1)`; còn ở đây hỏi "có bao nhiêu số **trong một khoảng**" → `dict` bó tay, vì khoảng có thể chứa rất nhiều giá trị khác nhau và ta không thể tra từng giá trị.

In [10]:
def show_windows(nums, lower, upper):
    P = build_prefix(nums)
    total = 0
    for b in range(1, len(P)):
        lo_val, hi_val = P[b] - upper, P[b] - lower
        good = [a for a in range(b) if lo_val <= P[a] <= hi_val]
        total += len(good)
        print(f"b={b}, P[b]={P[b]:>2}, cửa sổ=[{lo_val:>2}, {hi_val:>2}], "
              f"P đã gặp={P[:b]}, a hợp lệ={good} -> +{len(good)}")
    print("Tổng =", total)

show_windows([-2, 5, -1], -2, 2)

b=1, P[b]=-2, cửa sổ=[-4,  0], P đã gặp=[0], a hợp lệ=[0] -> +1
b=2, P[b]= 3, cửa sổ=[ 1,  5], P đã gặp=[0, -2], a hợp lệ=[] -> +0
b=3, P[b]= 2, cửa sổ=[ 0,  4], P đã gặp=[0, -2, 3], a hợp lệ=[0, 2] -> +2
Tổng = 3


### Câu hỏi thiết kế: cần một "túi" chứa các `P` đã gặp

Từ đây, mọi lời giải nhanh đều xoay quanh một "túi" (cấu trúc dữ liệu) chứa các prefix sum đã gặp, hỗ trợ hai thao tác:

1. **`thêm(x)`** — bỏ `P[b]` vào túi sau khi xử lý xong `b`.
2. **`đếm_trong_khoảng(L, R)`** — có bao nhiêu số trong túi nằm trong `[L, R]`.

Nếu cả hai thao tác đều `O(log n)` thì cả bài là `O(n log n)`. Ba hướng ta sẽ học:

| Cách | Ý tưởng | `thêm` | `đếm` | Tổng |
|---|---|---|---|---|
| **3** — list sắp xếp + `bisect` | Giữ list luôn sắp xếp, tìm nhị phân hai đầu cửa sổ | `O(n)` (dịch phần tử) | `O(log n)` | `O(n²)` worst, nhưng nhanh trong thực tế |
| **4** — merge sort (cách chính) | Không hỏi từng `b` nữa: chia đôi, đếm **cả loạt** cặp chéo khi trộn | — | — | `O(n log n)` |
| **5** — Fenwick Tree + nén tọa độ | Mảng đếm theo "hạng", tổng tiền tố trên mảng đếm | `O(log n)` | `O(log n)` | `O(n log n)` |

## 7. Kiến thức nền 2 — Tìm kiếm nhị phân (binary search) và module `bisect`

Trong một list **đã sắp xếp tăng dần**, ta có thể tìm "vị trí của `x`" mà không cần duyệt từng phần tử: nhìn phần tử ở giữa, nếu nó nhỏ hơn `x` thì bỏ cả nửa trái, ngược lại bỏ nửa phải. Mỗi bước loại một nửa → chỉ cần khoảng `log2(n)` bước (với `n = 10^5` là khoảng 17 bước). Đó là **binary search**.

Python có sẵn module `bisect` với hai hàm quan trọng (cho list `a` đã sắp xếp):

- `bisect_left(a, x)` = vị trí **đầu tiên** mà `a[pos] >= x` = **số phần tử `< x`**.
- `bisect_right(a, x)` = vị trí **đầu tiên** mà `a[pos] > x` = **số phần tử `<= x`**.

Ví dụ `a = [-2, 0, 0, 3, 5]`:

| x | `bisect_left(a, x)` | `bisect_right(a, x)` | Giải thích |
|:---:|:---:|:---:|---|
| 0 | 1 | 3 | 1 số `< 0` (là -2); 3 số `<= 0` (-2, 0, 0) |
| 1 | 3 | 3 | không có số 1 → hai hàm trùng nhau |
| -5 | 0 | 0 | nhỏ hơn mọi phần tử |
| 9 | 5 | 5 | lớn hơn mọi phần tử |

**Công thức đếm số phần tử trong `[L, R]`:**

```python
bisect_right(a, R) - bisect_left(a, L)     # (số phần tử <= R) - (số phần tử < L)
```

Ví dụ `[0, 3]`: `bisect_right(a, 3) − bisect_left(a, 0) = 4 − 1 = 3` (các số `0, 0, 3`). Nhớ: **cận trên dùng `bisect_right`** (để lấy cả các số bằng `R`), **cận dưới dùng `bisect_left`** (để lấy cả các số bằng `L`).

Thêm một hàm nữa: `insort(a, x)` chèn `x` vào đúng chỗ để `a` vẫn sắp xếp. Cell dưới còn tự cài `my_bisect_left` để thấy binary search chạy bên trong: giữ khoảng `[lo, hi)`, mọi phần tử trước `lo` chắc chắn `< x`, mọi phần tử từ `hi` trở đi chắc chắn `>= x`; thu hẹp tới khi `lo == hi`.

In [11]:
from bisect import bisect_left, bisect_right, insort

a = [-2, 0, 0, 3, 5]
for x in [0, 1, -5, 9]:
    print(f"x={x:>2}: bisect_left={bisect_left(a, x)} (số phần tử < x), "
          f"bisect_right={bisect_right(a, x)} (số phần tử <= x)")
L, R = 0, 3
print(f"Số phần tử trong [{L}, {R}] =", bisect_right(a, R) - bisect_left(a, L))

def my_bisect_left(a, x, verbose=False):
    lo, hi = 0, len(a)              # đáp án nằm trong [lo, hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1            # a[mid] còn nhỏ -> đáp án nằm bên phải mid
        else:
            hi = mid                # a[mid] >= x -> mid có thể là đáp án, bỏ nửa phải
        if verbose:
            print(f"   mid={mid}, a[mid]={a[mid]:>2} -> lo={lo}, hi={hi}")
    return lo

print("\nTự cài: tìm bisect_left(a, 3)")
print("Kết quả:", my_bisect_left(a, 3, verbose=True), "| thư viện:", bisect_left(a, 3))

bag = []
for x in [3, -2, 0, 5, 0]:
    insort(bag, x)
    print(f"insort {x:>2} -> {bag}")

x= 0: bisect_left=1 (số phần tử < x), bisect_right=3 (số phần tử <= x)
x= 1: bisect_left=3 (số phần tử < x), bisect_right=3 (số phần tử <= x)
x=-5: bisect_left=0 (số phần tử < x), bisect_right=0 (số phần tử <= x)
x= 9: bisect_left=5 (số phần tử < x), bisect_right=5 (số phần tử <= x)
Số phần tử trong [0, 3] = 3

Tự cài: tìm bisect_left(a, 3)
   mid=2, a[mid]= 0 -> lo=3, hi=5
   mid=4, a[mid]= 5 -> lo=3, hi=4
   mid=3, a[mid]= 3 -> lo=3, hi=3
Kết quả: 3 | thư viện: 3
insort  3 -> [3]
insort -2 -> [-2, 3]
insort  0 -> [-2, 0, 3]
insort  5 -> [-2, 0, 3, 5]
insort  0 -> [-2, 0, 0, 3, 5]


### `insort` có thật sự nhanh không?

`insort` gồm 2 việc: (1) **tìm vị trí** bằng binary search — `O(log n)`; (2) **chèn** vào vị trí đó. List của Python là một **mảng liên tiếp trong bộ nhớ**, nên chèn vào giữa phải **dịch tất cả phần tử phía sau** sang phải một ô — tốn `O(n)`. Ví dụ chèn `-2` vào `[3]` ở cell trên: số `3` phải dịch sang phải.

Ghi nhớ điều này, ta xây Cách 3 rồi đo xem việc dịch phần tử tốn bao nhiêu.

## 8. Cách 3 — Duyệt `b`, giữ list `seen` đã sắp xếp + `bisect`

Ghép "cửa sổ" ở mục 6 với `bisect`:

```python
seen = [0]             # túi chứa P[0] = 0, luôn sắp xếp tăng dần
prefix = 0
count = 0
for x in nums:
    prefix += x        # prefix bây giờ chính là P[b]
    # đếm các P[a] đã gặp nằm trong [prefix - upper, prefix - lower]
    count += bisect_right(seen, prefix - lower) - bisect_left(seen, prefix - upper)
    insort(seen, prefix)   # thêm P[b] vào túi SAU KHI đã đếm
```

Ba điểm cần hiểu:

1. **Không cần mảng `P` riêng**: biến `prefix` "lăn" dần, mỗi vòng cộng thêm `x` là ra `P[b]` mới (đúng tinh thần running sum ở Cách 1b).
2. **`seen` khởi tạo bằng `[0]`** chính là `P[0] = 0` — để đếm được các đoạn bắt đầu từ index 0.
3. **Đếm trước, thêm sau** — giống "kiểm tra trước, lưu sau" của Two Sum. Nếu thêm `P[b]` vào túi trước rồi mới đếm, ta sẽ đếm cả cặp `a = b` (đoạn rỗng, tổng `0`) mỗi khi `0 ∈ [lower, upper]`.

In [12]:
def count_range_sum_bisect(nums: List[int], lower: int, upper: int) -> int:
    seen = [0]                      # P[0] = 0
    prefix = 0
    count = 0
    for x in nums:
        prefix += x                 # P[b]
        count += bisect_right(seen, prefix - lower) - bisect_left(seen, prefix - upper)
        insort(seen, prefix)        # thêm sau khi đếm
    return count

for nums, lower, upper, expected in examples:
    got = count_range_sum_bisect(nums, lower, upper)
    print(nums, lower, upper, "->", got, "| expected:", expected)
    assert got == expected

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


### Trace cách 3

**Example 1** — `nums = [-2, 5, -1]`, `lower = -2`, `upper = 2`:

| b | `x` | `P[b]` | Cửa sổ `[P−upper, P−lower]` | `seen` trước | Số đếm được | `count` | `seen` sau |
|:---:|---:|---:|:---:|:---|:---|:---:|:---|
| 1 | -2 | -2 | `[-4, 0]` | `[0]` | 1 (số `0`) | 1 | `[-2, 0]` |
| 2 | 5 | 3 | `[1, 5]` | `[-2, 0]` | 0 | 1 | `[-2, 0, 3]` |
| 3 | -1 | 2 | `[0, 4]` | `[-2, 0, 3]` | 2 (số `0`, `3`) | 3 | `[-2, 0, 2, 3]` |

**Ví dụ có giá trị trùng** — `nums = [1, -1, 1, -1]`, `lower = upper = 0` (đếm đoạn có tổng đúng bằng 0). `P = [0, 1, 0, 1, 0]`. Ở `b = 4`: `P[4] = 0`, cửa sổ `[0, 0]`, `seen = [0, 0, 1, 1]` → `bisect_right(seen, 0) − bisect_left(seen, 0) = 2 − 0 = 2`: **cả hai số 0 đều được đếm** nhờ cặp `bisect_left`/`bisect_right` "kẹp" trọn khối giá trị bằng nhau. Tổng là **4** đoạn: `[1,-1]` ở vị trí 0–1, 1–2, 2–3 và cả mảng 0–3.

In [13]:
def trace_bisect(nums, lower, upper):
    seen, prefix, count = [0], 0, 0
    print(f"nums={nums}, lower={lower}, upper={upper}; bắt đầu seen={seen}")
    for b, x in enumerate(nums, start=1):         # enumerate(..., start=1): b chạy 1, 2, 3, ...
        prefix += x
        L, R = prefix - upper, prefix - lower
        left_pos, right_pos = bisect_left(seen, L), bisect_right(seen, R)
        got = right_pos - left_pos
        count += got
        print(f"  b={b}, x={x:>2}, P[b]={prefix:>2}, cửa sổ=[{L:>2}, {R:>2}], seen={seen}, "
              f"right={right_pos}, left={left_pos} -> +{got}, count={count}")
        insort(seen, prefix)
    print(f"  kết thúc: seen={seen}, count={count}")
    return count

result_1 = trace_bisect([-2, 5, -1], -2, 2)
print()
result_2 = trace_bisect([1, -1, 1, -1], 0, 0)

nums=[-2, 5, -1], lower=-2, upper=2; bắt đầu seen=[0]
  b=1, x=-2, P[b]=-2, cửa sổ=[-4,  0], seen=[0], right=1, left=0 -> +1, count=1
  b=2, x= 5, P[b]= 3, cửa sổ=[ 1,  5], seen=[-2, 0], right=2, left=2 -> +0, count=1
  b=3, x=-1, P[b]= 2, cửa sổ=[ 0,  4], seen=[-2, 0, 3], right=3, left=1 -> +2, count=3
  kết thúc: seen=[-2, 0, 2, 3], count=3

nums=[1, -1, 1, -1], lower=0, upper=0; bắt đầu seen=[0]
  b=1, x= 1, P[b]= 1, cửa sổ=[ 1,  1], seen=[0], right=1, left=1 -> +0, count=0
  b=2, x=-1, P[b]= 0, cửa sổ=[ 0,  0], seen=[0, 1], right=1, left=0 -> +1, count=1
  b=3, x= 1, P[b]= 1, cửa sổ=[ 1,  1], seen=[0, 0, 1], right=3, left=2 -> +1, count=2
  b=4, x=-1, P[b]= 0, cửa sổ=[ 0,  0], seen=[0, 0, 1, 1], right=2, left=0 -> +2, count=4
  kết thúc: seen=[0, 0, 0, 1, 1], count=4


### Độ phức tạp cách 3 — và cái giá ẩn của `insort`

Gọi `n = len(nums)`.

- Mỗi vòng có 2 lần `bisect` → `O(log n)`.
- Mỗi vòng có 1 lần `insort` → tìm chỗ `O(log n)` nhưng **dịch phần tử `O(n)`**.
- **Thời gian worst case: `O(n²)`** (ví dụ prefix sum giảm dần → lần nào cũng chèn vào đầu list, dịch toàn bộ). **Bộ nhớ: `O(n)`** cho `seen`.

Cell dưới đếm tổng số phần tử bị dịch: với dữ liệu ngẫu nhiên khoảng `n²/4`, với dữ liệu giảm dần khoảng `n²/2` — gấp đôi `n` thì gấp bốn số lần dịch, đúng dáng **bậc hai**.

**Vì sao trong thực tế cách 3 vẫn khá nhanh?** Việc dịch phần tử được làm bằng một lệnh chép bộ nhớ (memmove) viết bằng C, cực nhanh so với vòng lặp Python. Nhưng đó là "may nhờ hằng số nhỏ", **không phải đảm bảo** `O(n log n)`.

Muốn `thêm` thật sự `O(log n)` thì cần **cây tìm kiếm cân bằng** (balanced BST: AVL, red-black tree, treap...) hoặc `SortedList` của thư viện `sortedcontainers` (LeetCode có cài sẵn, nhưng **không** thuộc thư viện chuẩn Python). Hai cách tiếp theo đạt `O(n log n)` chỉ bằng thư viện chuẩn.

In [14]:
import random

def count_insort_shifts(values):
    seen, shifts = [], 0
    for v in values:
        pos = bisect_right(seen, v)
        shifts += len(seen) - pos          # số phần tử phải dịch sang phải khi chèn vào pos
        seen.insert(pos, v)
    return shifts

random.seed(327)
for n in [1000, 2000, 4000]:
    random_vals = [random.randint(-10**9, 10**9) for _ in range(n)]
    decreasing = list(range(n, 0, -1))     # giảm dần: lần nào cũng chèn vào đầu
    print(f"n={n}: ngẫu nhiên dịch {count_insort_shifts(random_vals):>9,} | "
          f"giảm dần dịch {count_insort_shifts(decreasing):>9,} | n²/2 = {n * n // 2:>9,}")

n=1000: ngẫu nhiên dịch   241,183 | giảm dần dịch   499,500 | n²/2 =   500,000
n=2000: ngẫu nhiên dịch 1,021,565 | giảm dần dịch 1,999,000 | n²/2 = 2,000,000
n=4000: ngẫu nhiên dịch 4,006,096 | giảm dần dịch 7,998,000 | n²/2 = 8,000,000


## 9. Cách 4 — Insight "chia để trị": đếm cả loạt cặp chéo

Cách 3 trả lời từng câu hỏi "với `b` này, có bao nhiêu `P[a]` trong cửa sổ?" một cách **online** (lần lượt), nên cần một túi vừa thêm nhanh vừa đếm nhanh. Cách 4 đổi góc nhìn: ta có **toàn bộ** mảng `P` ngay từ đầu, vậy hãy đếm **hàng loạt**.

**Divide and conquer (chia để trị):** chia `P` (dài `N = n + 1`) tại `mid` thành nửa trái `P[0..mid−1]` và nửa phải `P[mid..N−1]`. Mọi cặp `(a, b)` với `a < b` rơi vào **đúng một** trong 3 loại:

1. `a` và `b` **cùng ở nửa trái** → bài toán con bên trái (giải bằng đệ quy).
2. `a` và `b` **cùng ở nửa phải** → bài toán con bên phải (đệ quy).
3. `a` ở **trái**, `b` ở **phải** → **cặp chéo**.

**Phép màu nằm ở loại 3:** mọi chỉ số bên trái đều nhỏ hơn mọi chỉ số bên phải, nên điều kiện `a < b` **tự động đúng**. Ta chỉ cần biết "giá trị này thuộc nửa trái hay nửa phải", **không cần biết thứ tự bên trong mỗi nửa**. Vì vậy ta được phép **sắp xếp từng nửa** — và khi hai nửa đã sắp xếp, đếm cặp chéo chỉ tốn thời gian tuyến tính bằng **two pointers**. Việc sắp xếp thì merge sort làm "miễn phí" trong lúc đệ quy.

Ví dụ `P = [0, -2, 3, 2]` → trái `[0, -2]`, phải `[3, 2]`:

- Loại 1: cặp `(0, 1)`: `-2 − 0 = -2` ✅ → 1
- Loại 2: cặp `(2, 3)`: `2 − 3 = -1` ✅ → 1
- Loại 3: 4 cặp chéo, chỉ có `(0, 3)`: `2 − 0 = 2` ✅ → 1

Tổng **3**. Ý tưởng này giống hệt bài kinh điển **"đếm nghịch thế"** (count inversions) bằng merge sort. Để cài được, ta cần 3 viên gạch: đệ quy chia đôi, trộn hai mảng đã sắp xếp, và two pointers đếm cặp chéo.

## 10. Kiến thức nền 3 — Đệ quy (recursion) chia đôi với khoảng nửa mở `[lo, hi)`

**Đệ quy** là khi một hàm **gọi lại chính nó** trên bài toán **nhỏ hơn**. Mọi hàm đệ quy cần:

1. **Base case** (trường hợp cơ sở): bài toán đủ nhỏ để trả lời ngay, **không gọi tiếp** — nếu thiếu, hàm gọi mãi không dừng.
2. **Bước đệ quy**: chia thành bài nhỏ hơn, gọi chính hàm đó, rồi ghép kết quả.

Ta dùng **khoảng nửa mở** `[lo, hi)`: gồm `lo`, **không gồm** `hi` — giống `range(lo, hi)` và slicing `a[lo:hi]`. Ưu điểm: độ dài là `hi − lo`, và chia tại `mid` cho hai nửa `[lo, mid)` và `[mid, hi)` **khít nhau, không trùng, không hở**.

Ví dụ nhỏ: tính tổng `a[lo:hi]` bằng cách chia đôi. Cell dưới in "cây gọi hàm" — dòng thụt vào càng sâu là lời gọi càng sâu. Hãy để ý: mỗi lời gọi **chờ** hai lời gọi con trả về rồi mới tính tiếp.

In [15]:
def sum_divide(a, lo, hi, depth=0):
    """Tổng a[lo:hi] bằng chia đôi (hi - lo >= 1)."""
    pad = "    " * depth
    if hi - lo == 1:                                  # base case: chỉ 1 phần tử
        print(f"{pad}sum_divide({lo},{hi}) = {a[lo]}   (base case)")
        return a[lo]
    mid = (lo + hi) // 2
    print(f"{pad}sum_divide({lo},{hi}): chia thành [{lo},{mid}) và [{mid},{hi})")
    total = sum_divide(a, lo, mid, depth + 1) + sum_divide(a, mid, hi, depth + 1)
    print(f"{pad}=> sum_divide({lo},{hi}) = {total}")
    return total

total = sum_divide([3, 1, 4, 1], 0, 4)

sum_divide(0,4): chia thành [0,2) và [2,4)
    sum_divide(0,2): chia thành [0,1) và [1,2)
        sum_divide(0,1) = 3   (base case)
        sum_divide(1,2) = 1   (base case)
    => sum_divide(0,2) = 4
    sum_divide(2,4): chia thành [2,3) và [3,4)
        sum_divide(2,3) = 4   (base case)
        sum_divide(3,4) = 1   (base case)
    => sum_divide(2,4) = 5
=> sum_divide(0,4) = 9


## 11. Kiến thức nền 4 — Trộn (merge) hai list đã sắp xếp, và merge sort

**Bài toán trộn:** có `left = [1, 4, 6]` và `right = [2, 3, 7]`, cả hai đã sắp xếp. Muốn có một list sắp xếp chứa tất cả.

Dùng hai con trỏ `i` (trên `left`) và `j` (trên `right`): mỗi bước so sánh `left[i]` với `right[j]`, lấy số **nhỏ hơn** cho vào kết quả và tiến con trỏ đó. Khi một bên hết, chép nốt phần còn dư của bên kia.

```text
i→1  j→2 : lấy 1        kết quả [1]
i→4  j→2 : lấy 2        [1, 2]
i→4  j→3 : lấy 3        [1, 2, 3]
i→4  j→7 : lấy 4        [1, 2, 3, 4]
i→6  j→7 : lấy 6        [1, 2, 3, 4, 6]
left hết  : chép [7]    [1, 2, 3, 4, 6, 7]
```

Mỗi phần tử được lấy đúng một lần → trộn tốn `O(len(left) + len(right))`.

**Merge sort** = đệ quy: chia đôi list, **sắp xếp mỗi nửa bằng chính merge sort**, rồi **trộn** hai nửa. Base case: list 0 hoặc 1 phần tử đã sắp xếp sẵn. Cây đệ quy có khoảng `log2 n` tầng, mỗi tầng trộn tổng cộng `n` phần tử → `O(n log n)`.

In [16]:
def merge_two_sorted(left, right):
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])          # phần còn dư của left (nếu có)
    merged.extend(right[j:])         # phần còn dư của right (nếu có)
    return merged

def merge_sort(a):
    if len(a) <= 1:                  # base case
        return a[:]
    mid = len(a) // 2
    return merge_two_sorted(merge_sort(a[:mid]), merge_sort(a[mid:]))

print("Trộn [1, 4, 6] và [2, 3, 7]:", merge_two_sorted([1, 4, 6], [2, 3, 7]))
print("merge_sort([5, 2, 4, 1, 3]) :", merge_sort([5, 2, 4, 1, 3]))
print("merge_sort([0, -2, 3, 2])   :", merge_sort([0, -2, 3, 2]))

Trộn [1, 4, 6] và [2, 3, 7]: [1, 2, 3, 4, 6, 7]
merge_sort([5, 2, 4, 1, 3]) : [1, 2, 3, 4, 5]
merge_sort([0, -2, 3, 2])   : [-2, 0, 2, 3]


## 12. Kiến thức nền 5 — Two pointers đếm cặp chéo trên hai list đã sắp xếp

**Bài toán nhỏ:** `left` chứa các giá trị `P[a]` (nửa trái), `right` chứa các giá trị `P[b]` (nửa phải), **cả hai đã sắp xếp tăng dần**. Đếm số cặp `(x ∈ left, y ∈ right)` sao cho `lower ≤ y − x ≤ upper`, tức là (chuyển vế như mục 6) `x ∈ [y − upper, y − lower]`.

Duyệt `y` trong `right` theo thứ tự **tăng dần**. Khi `y` tăng, **cả hai đầu cửa sổ đều tăng** → hai con trỏ trên `left` **chỉ tiến, không bao giờ lùi**:

- `lo` = vị trí đầu tiên mà `left[lo] >= y − upper` (các số trước `lo` quá nhỏ: hiệu `> upper`).
- `hi` = vị trí đầu tiên mà `left[hi] > y − lower` (các số từ `hi` trở đi quá lớn: hiệu `< lower`).
- Số cặp với `y` này là `hi − lo` (các số `left[lo:hi]`).

Đây chính là `bisect_left`/`bisect_right` của Cách 3, nhưng nhờ `y` tăng dần nên **không phải tìm lại từ đầu** — chỉ cần trượt tiếp. Mỗi con trỏ đi tối đa `len(left)` bước → tổng `O(len(left) + len(right))`.

Ví dụ `left = [-2, 0, 4]`, `right = [0, 1, 2]`, `lower = -1`, `upper = 2`:

| y | Cửa sổ `[y−upper, y−lower]` | `lo` | `hi` | `left[lo:hi]` | + cặp |
|:---:|:---:|:---:|:---:|:---|:---:|
| 0 | `[-2, 1]` | 0 | 2 | `[-2, 0]` | 2 |
| 1 | `[-1, 2]` | 1 | 2 | `[0]` | 1 |
| 2 | `[0, 3]` | 1 | 2 | `[0]` | 1 |

Tổng **4** cặp. Cell dưới kiểm chứng bằng cách thử mọi cặp.

In [17]:
def count_cross_pairs(left, right, lower, upper, verbose=False):
    """left, right đã sắp xếp tăng dần. Đếm cặp (x thuộc left, y thuộc right) với lower <= y - x <= upper."""
    lo = hi = 0
    count = 0
    for y in right:                                    # y tăng dần
        while lo < len(left) and left[lo] < y - upper:
            lo += 1                                    # left[lo] quá nhỏ: y - left[lo] > upper
        while hi < len(left) and left[hi] <= y - lower:
            hi += 1                                    # left[hi] vẫn đủ nhỏ: y - left[hi] >= lower
        count += hi - lo
        if verbose:
            print(f"y={y}: cửa sổ=[{y - upper:>2}, {y - lower:>2}], lo={lo}, hi={hi}, "
                  f"left[lo:hi]={left[lo:hi]} -> +{hi - lo}")
    return count

left, right = [-2, 0, 4], [0, 1, 2]
print("Two pointers:", count_cross_pairs(left, right, -1, 2, verbose=True))
print("Thử mọi cặp :", sum(-1 <= y - x <= 2 for x in left for y in right))

y=0: cửa sổ=[-2,  1], lo=0, hi=2, left[lo:hi]=[-2, 0] -> +2
y=1: cửa sổ=[-1,  2], lo=1, hi=2, left[lo:hi]=[0] -> +1
y=2: cửa sổ=[ 0,  3], lo=1, hi=2, left[lo:hi]=[0] -> +1
Two pointers: 4
Thử mọi cặp : 4


## 13. Ghép lại: Cách 4 — Merge sort đếm khi trộn (cách chính)

Định nghĩa hàm đệ quy `sort_count(lo, hi)` làm **hai việc cùng lúc** trên đoạn `P[lo:hi]`:

- **Trả về** số cặp `(a, b)` với `lo ≤ a < b < hi` và `lower ≤ P[b] − P[a] ≤ upper`.
- **Sắp xếp** `P[lo:hi]` tăng dần (để lời gọi cha dùng được two pointers).

Các bước:

1. **Base case:** `hi − lo <= 1` (0 hoặc 1 phần tử) → không có cặp nào, trả `0`.
2. `mid = (lo + hi) // 2`; gọi `sort_count(lo, mid)` và `sort_count(mid, hi)` → đếm xong loại 1 và loại 2, đồng thời hai nửa **đã được sắp xếp**.
3. **Đếm cặp chéo** (loại 3) bằng two pointers: duyệt `b` trong `[mid, hi)`, con trỏ `i`, `j` chạy trong `[lo, mid)`.
4. **Trộn** hai nửa thành `P[lo:hi]` đã sắp xếp.
5. Trả về tổng ba loại.

⚠️ **Thứ tự bước 3 → bước 4 là bắt buộc**: phải **đếm khi hai nửa còn tách riêng**. Sau khi trộn, các giá trị trái và phải lẫn vào nhau, ta không còn biết số nào "đến từ bên trái" (tức là có chỉ số nhỏ hơn) nữa.

Đáp án là `sort_count(0, len(P))`. Cài đặt dưới đây sửa trực tiếp trên list `P` bằng chỉ số `lo`, `hi` (không cắt list con khi đệ quy), chỉ dùng list tạm `merged` khi trộn.

In [18]:
def count_range_sum_merge_sort(nums: List[int], lower: int, upper: int) -> int:
    P = build_prefix(nums)

    def sort_count(lo: int, hi: int) -> int:
        if hi - lo <= 1:                               # base case: 0 hoặc 1 phần tử
            return 0
        mid = (lo + hi) // 2
        count = sort_count(lo, mid) + sort_count(mid, hi)   # loại 1 + loại 2; hai nửa giờ đã sắp xếp

        # Loại 3: cặp chéo a trong [lo, mid), b trong [mid, hi)
        i = j = lo
        for b in range(mid, hi):                       # P[b] tăng dần
            while i < mid and P[i] < P[b] - upper:
                i += 1
            while j < mid and P[j] <= P[b] - lower:
                j += 1
            count += j - i

        # Trộn hai nửa đã sắp xếp
        merged = []
        p1, p2 = lo, mid
        while p1 < mid and p2 < hi:
            if P[p1] <= P[p2]:
                merged.append(P[p1])
                p1 += 1
            else:
                merged.append(P[p2])
                p2 += 1
        merged.extend(P[p1:mid])
        merged.extend(P[p2:hi])
        P[lo:hi] = merged                              # ghi đè: P[lo:hi] đã sắp xếp
        return count

    return sort_count(0, len(P))

for nums, lower, upper, expected in examples:
    got = count_range_sum_merge_sort(nums, lower, upper)
    print(nums, lower, upper, "->", got, "| expected:", expected)
    assert got == expected

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


### Trace cách 4 trên Example 1 — từng nút của cây đệ quy

`P = [0, -2, 3, 2]` (chỉ số 0..3). Cây đệ quy (mỗi nút là một khoảng `[lo, hi)`):

```text
[0,4)  P = [0, -2, 3, 2]
├── [0,2)  [0, -2]
│   ├── [0,1)  [0]     base case
│   └── [1,2)  [-2]    base case
└── [2,4)  [3, 2]
    ├── [2,3)  [3]     base case
    └── [3,4)  [2]     base case
```

| Nút | Trái (đã sắp) | Phải (đã sắp) | Cặp chéo hợp lệ `(P[a], P[b])` | + chéo | Sau khi trộn |
|:---:|:---|:---|:---|:---:|:---|
| `[0,2)` | `[0]` | `[-2]` | `(0, -2)`: `-2 − 0 = -2` ✅ → đoạn `[0..0]` | 1 | `[-2, 0]` |
| `[2,4)` | `[3]` | `[2]` | `(3, 2)`: `2 − 3 = -1` ✅ → đoạn `[2..2]` | 1 | `[2, 3]` |
| `[0,4)` | `[-2, 0]` | `[2, 3]` | `(0, 2)`: `2 − 0 = 2` ✅ → đoạn `[0..2]` | 1 | `[-2, 0, 2, 3]` |

Tổng `1 + 1 + 1 = 3`, và ba cặp tìm được ứng với **đúng ba đoạn** `[0,0]`, `[2,2]`, `[0,2]` mà đề giải thích!

Ở nút gốc, nửa trái **đã bị sắp xếp thành** `[-2, 0]` (thứ tự gốc là `[0, -2]`) nhưng kết quả vẫn đúng — vì với cặp chéo, ta chỉ cần biết số nào ở bên trái.

Cell dưới in trace thật (hàm trace mang kèm **chỉ số gốc** của mỗi giá trị để in ra "đoạn nào" — bản chính không cần). Dòng in ra theo thứ tự **hoàn thành**: nút con xong trước nút cha, thụt vào càng sâu là nút càng sâu. Ví dụ thứ hai `nums = [4, -6, 3, 1, -2]`, `lower = -1`, `upper = 2` có 6 đoạn hợp lệ, trong đó nút gốc đếm được 4 cặp chéo cùng lúc.

In [19]:
def trace_merge_sort(nums, lower, upper):
    P = build_prefix(nums)
    items = [(v, idx) for idx, v in enumerate(P)]      # (giá trị, chỉ số gốc) để in cho dễ hiểu
    print(f"nums={nums}, lower={lower}, upper={upper}, P={P}")

    def sort_count(lo, hi, depth):
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        count = sort_count(lo, mid, depth + 1) + sort_count(mid, hi, depth + 1)
        left = [v for v, _ in items[lo:mid]]
        right = [v for v, _ in items[mid:hi]]
        i = j = lo
        found = []
        for b in range(mid, hi):
            vb, ib = items[b]
            while i < mid and items[i][0] < vb - upper:
                i += 1
            while j < mid and items[j][0] <= vb - lower:
                j += 1
            found += [f"đoạn[{ia}..{ib - 1}]={vb - va}" for va, ia in items[i:j]]
        count += len(found)
        items[lo:hi] = sorted(items[lo:hi])           # trace dùng sorted() cho gọn
        pad = "    " * depth
        print(f"{pad}[{lo},{hi}) trái={left} phải={right} -> chéo +{len(found)} {found}")
        print(f"{pad}       sau khi trộn: {[v for v, _ in items[lo:hi]]}")
        return count

    total = sort_count(0, len(P), 0)
    print("Tổng số đoạn hợp lệ =", total)
    return total

result_1 = trace_merge_sort([-2, 5, -1], -2, 2)
print()
result_2 = trace_merge_sort([4, -6, 3, 1, -2], -1, 2)

nums=[-2, 5, -1], lower=-2, upper=2, P=[0, -2, 3, 2]
    [0,2) trái=[0] phải=[-2] -> chéo +1 ['đoạn[0..0]=-2']
           sau khi trộn: [-2, 0]
    [2,4) trái=[3] phải=[2] -> chéo +1 ['đoạn[2..2]=-1']
           sau khi trộn: [2, 3]
[0,4) trái=[-2, 0] phải=[2, 3] -> chéo +1 ['đoạn[0..2]=2']
       sau khi trộn: [-2, 0, 2, 3]
Tổng số đoạn hợp lệ = 3

nums=[4, -6, 3, 1, -2], lower=-1, upper=2, P=[0, 4, -2, 1, 2, 0]
        [1,3) trái=[4] phải=[-2] -> chéo +0 []
               sau khi trộn: [-2, 4]
    [0,3) trái=[0] phải=[-2, 4] -> chéo +0 []
           sau khi trộn: [-2, 0, 4]
        [4,6) trái=[2] phải=[0] -> chéo +0 []
               sau khi trộn: [0, 2]
    [3,6) trái=[1] phải=[0, 2] -> chéo +2 ['đoạn[3..4]=-1', 'đoạn[3..3]=1']
           sau khi trộn: [0, 1, 2]
[0,6) trái=[-2, 0, 4] phải=[0, 1, 2] -> chéo +4 ['đoạn[2..4]=2', 'đoạn[0..4]=0', 'đoạn[0..2]=1', 'đoạn[0..3]=2']
       sau khi trộn: [-2, 0, 0, 1, 2, 4]
Tổng số đoạn hợp lệ = 6


### Vì sao cách 4 đúng? Và vì sao nó `O(n log n)`?

**Không bỏ sót, không đếm trùng.** Xét một cặp bất kỳ `(a, b)` với `a < b`. Đi từ nút gốc xuống: chừng nào `a` và `b` còn cùng nằm trong một nửa, ta đi tiếp xuống nửa đó. Sẽ có **đúng một nút** đầu tiên mà `a` rơi vào nửa trái còn `b` rơi vào nửa phải — nút đó (và chỉ nút đó) đếm cặp `(a, b)` ở bước "cặp chéo". Ở các nút cha phía trên, `a` và `b` nằm cùng một nửa nên không được đếm là cặp chéo; ở các nút con phía dưới, `a` và `b` không còn cùng một nút.

**Sắp xếp không làm sai.** Invariant (bất biến): *khi `sort_count(lo, hi)` trả về, `P[lo:hi]` chứa đúng các giá trị ban đầu của vùng đó, chỉ đổi thứ tự thành tăng dần.* Nên tập giá trị của nửa trái vẫn là `{P[a] : lo ≤ a < mid}` — đúng những thứ cần để đếm cặp chéo.

**Two pointers đúng** vì `P[b]` tăng dần nên cả hai ngưỡng `P[b] − upper` và `P[b] − lower` tăng dần; các con trỏ chỉ cần tiến.

**Độ phức tạp.** Gọi `N = n + 1` là độ dài `P`. Mỗi nút độ dài `m` làm `O(m)` việc (đếm chéo + trộn) ngoài hai lời gọi con: `T(N) = 2·T(N/2) + O(N)`. Cây đệ quy có khoảng `log2 N` tầng, mỗi tầng tổng độ dài các nút đúng bằng `N` → mỗi tầng `O(N)`:

| n | `n²/2` (Cách 1, 2) | `n · log2 n` (Cách 4) |
|---:|---:|---:|
| 1 000 | 500 000 | ≈ 10 000 |
| 100 000 | 5 000 000 000 | ≈ 1 700 000 |

→ **Thời gian `O(n log n)`**. **Bộ nhớ `O(n)`**: mảng `P` + list tạm `merged` + ngăn xếp đệ quy sâu khoảng `log2 N ≈ 17` tầng (rất an toàn với giới hạn đệ quy mặc định 1000 của Python).

## 14. Cách 5 — Quay lại "túi online", lần này với mảng đếm

Cách 4 rất đẹp, nhưng nó cần đệ quy và "cắt" bài toán thành từng nút. Cách 5 giữ nguyên lối nghĩ **duyệt `b` từ trái sang phải** của Cách 3, chỉ thay "túi" list sắp xếp bằng một cái túi mà `thêm` cũng nhanh.

**Ý tưởng mảng đếm.** Giả sử mọi giá trị là số nguyên nhỏ trong `0..7`. Dùng mảng `cnt` với `cnt[v]` = số lần giá trị `v` đã được thêm:

- `thêm(x)`: `cnt[x] += 1` → `O(1)`.
- `đếm số phần tử ≤ R`: `cnt[0] + cnt[1] + ... + cnt[R]` → phải cộng `R + 1` ô, `O(V)` với `V` là số giá trị có thể có.
- `đếm trong [L, R]` = `(số ≤ R) − (số ≤ L − 1)`.

Hai vấn đề khi áp dụng vào bài:

1. **Giá trị quá lớn và có số âm**: `P[b]` có thể tới khoảng `±2·10^14` → không thể tạo mảng dài như vậy. → Giải bằng **nén tọa độ**.
2. **Đếm `O(V)` quá chậm**. → Giải bằng **Fenwick Tree**: cả `thêm` lẫn `đếm số ≤ R` đều `O(log V)`.

In [20]:
cnt = [0] * 8                          # đếm cho các giá trị 0..7
for x in [3, 5, 3, 0, 7]:
    cnt[x] += 1                        # thêm(x): O(1)
print("cnt =", cnt)

def count_at_most(cnt, R):             # số phần tử <= R: cộng R + 1 ô -> O(V)
    return sum(cnt[:R + 1])

print("Số phần tử <= 3       :", count_at_most(cnt, 3))                            # 0, 3, 3
print("Số phần tử trong [1, 5]:", count_at_most(cnt, 5) - count_at_most(cnt, 0))  # 3, 3, 5

cnt = [1, 0, 0, 2, 0, 1, 0, 1]
Số phần tử <= 3       : 3
Số phần tử trong [1, 5]: 3


## 15. Kiến thức nền 6 — Nén tọa độ (coordinate compression)

Bài toán chỉ quan tâm **so sánh lớn/nhỏ** giữa các giá trị (`P[a]` có nằm trong cửa sổ không), không quan tâm giá trị cụ thể lớn cỡ nào. Vậy ta thay mỗi giá trị bằng **thứ hạng** (rank) của nó trong danh sách các giá trị phân biệt đã sắp xếp:

```python
vals = sorted(set(P))                 # các giá trị phân biệt, tăng dần
rank(v) = bisect_left(vals, v) + 1    # hạng bắt đầu từ 1 (Fenwick dùng chỉ số từ 1)
```

Ví dụ `P = [0, -2, 3, 2]` → `vals = [-2, 0, 2, 3]` → hạng: `-2 → 1`, `0 → 2`, `2 → 3`, `3 → 4`. Dù giá trị gốc là `10^14` thì hạng cũng chỉ trong khoảng `1..N`.

**Cận cửa sổ không có trong `vals` thì sao?** Không cần thêm chúng vào. Ta dùng lại `bisect`:

- "Số giá trị đã thêm `≤ R`" = "số giá trị đã thêm có hạng `≤ bisect_right(vals, R)`" (vì `bisect_right(vals, R)` = số giá trị phân biệt `≤ R` = hạng lớn nhất còn `≤ R`).
- "Số giá trị đã thêm `< L`" = "số giá trị đã thêm có hạng `≤ bisect_left(vals, L)`".

Ví dụ: `R = 4` → `bisect_right(vals, 4) = 4` → mọi hạng `1..4` (đúng, cả 4 giá trị đều `≤ 4`). `L = 1` → `bisect_left(vals, 1) = 2` → hạng `1, 2` (giá trị `-2, 0`) là những số `< 1`. Chính xác!

In [21]:
P = [0, -2, 3, 2]
vals = sorted(set(P))
rank = {v: i + 1 for i, v in enumerate(vals)}       # hạng bắt đầu từ 1
print("vals =", vals)
print("rank =", rank)

big = [0, 2**31 - 1, -(2**31), 10**14, 2**31 - 1]
vals_big = sorted(set(big))
print("Giá trị lớn :", big)
print("Sau khi nén :", [bisect_left(vals_big, v) + 1 for v in big], "(giữ nguyên thứ tự lớn/nhỏ)")

for R in [4, -3, 2]:
    print(f"Số giá trị trong vals <= {R:>2}: bisect_right = {bisect_right(vals, R)}")
for L in [1, -2]:
    print(f"Số giá trị trong vals <  {L:>2}: bisect_left  = {bisect_left(vals, L)}")

vals = [-2, 0, 2, 3]
rank = {-2: 1, 0: 2, 2: 3, 3: 4}
Giá trị lớn : [0, 2147483647, -2147483648, 100000000000000, 2147483647]
Sau khi nén : [2, 3, 1, 4, 3] (giữ nguyên thứ tự lớn/nhỏ)
Số giá trị trong vals <=  4: bisect_right = 4
Số giá trị trong vals <= -3: bisect_right = 0
Số giá trị trong vals <=  2: bisect_right = 3
Số giá trị trong vals <   1: bisect_left  = 2
Số giá trị trong vals <  -2: bisect_left  = 0


## 16. Kiến thức nền 7 — Fenwick Tree (Binary Indexed Tree, BIT) từ con số `i & -i`

**Mục tiêu:** trên mảng đếm `cnt[1..m]` (chỉ số từ **1**), làm được hai việc trong `O(log m)`:

- `update(i)`: `cnt[i] += 1`
- `query(i)`: tính `cnt[1] + cnt[2] + ... + cnt[i]` (tổng tiền tố)

**Viên gạch: `lowbit(i) = i & -i`** — giá trị của **bit 1 thấp nhất** trong biểu diễn nhị phân của `i` (cũng là lũy thừa của 2 lớn nhất chia hết `i`). Ví dụ `6 = 0110₂` → `lowbit(6) = 0010₂ = 2`; `8 = 1000₂` → `lowbit(8) = 8`. Vì sao `i & -i` làm được điều đó? Python (và máy tính) biểu diễn `-i` dạng "bù hai": `-i = (đảo mọi bit của i) + 1`, nên mọi bit phía trên bit 1 thấp nhất bị đảo (AND ra 0), chỉ bit đó còn lại.

**Ý tưởng BIT:** mảng `tree[i]` lưu tổng của `cnt` trên đoạn **`(i − lowbit(i), i]`** — đoạn dài đúng `lowbit(i)` kết thúc tại `i`:

| i | Nhị phân | lowbit | `tree[i]` phụ trách |
|:---:|:---:|:---:|:---|
| 1 | 0001 | 1 | `cnt[1]` |
| 2 | 0010 | 2 | `cnt[1..2]` |
| 3 | 0011 | 1 | `cnt[3]` |
| 4 | 0100 | 4 | `cnt[1..4]` |
| 5 | 0101 | 1 | `cnt[5]` |
| 6 | 0110 | 2 | `cnt[5..6]` |
| 7 | 0111 | 1 | `cnt[7]` |
| 8 | 1000 | 8 | `cnt[1..8]` |

- **`query(7)`** = `cnt[1..7]` = `tree[7]` (`cnt[7]`) + `tree[6]` (`cnt[5..6]`) + `tree[4]` (`cnt[1..4]`). Đường đi `7 → 6 → 4 → 0`: mỗi bước `i -= lowbit(i)` (xóa bit 1 thấp nhất).
- **`update(3)`**: những ô nào phụ trách đoạn chứa `3`? `tree[3]`, `tree[4]`, `tree[8]`. Đường đi `3 → 4 → 8`: mỗi bước `i += lowbit(i)`.

Mỗi bước của `query` xóa một bit 1, mỗi bước của `update` làm bit thấp nhất "dịch lên" ít nhất một vị trí → cả hai tối đa khoảng `log2(m) + 1` bước.

In [22]:
for i in range(1, 9):
    low = i & -i
    print(f"i={i}  nhị phân={i:04b}  lowbit={low}  tree[{i}] phụ trách cnt[{i - low + 1}..{i}]")

i=1  nhị phân=0001  lowbit=1  tree[1] phụ trách cnt[1..1]
i=2  nhị phân=0010  lowbit=2  tree[2] phụ trách cnt[1..2]
i=3  nhị phân=0011  lowbit=1  tree[3] phụ trách cnt[3..3]
i=4  nhị phân=0100  lowbit=4  tree[4] phụ trách cnt[1..4]
i=5  nhị phân=0101  lowbit=1  tree[5] phụ trách cnt[5..5]
i=6  nhị phân=0110  lowbit=2  tree[6] phụ trách cnt[5..6]
i=7  nhị phân=0111  lowbit=1  tree[7] phụ trách cnt[7..7]
i=8  nhị phân=1000  lowbit=8  tree[8] phụ trách cnt[1..8]


### Cài đặt Fenwick Tree và kiểm chứng

```python
def update(i, delta=1):          # cnt[i] += delta
    while i <= m:
        tree[i] += delta
        i += i & -i              # sang ô "phụ trách đoạn lớn hơn" có chứa i

def query(i):                    # cnt[1] + ... + cnt[i]
    s = 0
    while i > 0:
        s += tree[i]
        i -= i & -i              # sang đoạn liền kề phía trước
    return s
```

Chú ý `query(0) = 0` (vòng `while` không chạy) — rất tiện khi cận dưới của cửa sổ nhỏ hơn mọi giá trị. Cell dưới in đường đi của `update`/`query` khớp với bảng trên, rồi so Fenwick với mảng đếm thường trên 200 thao tác ngẫu nhiên.

In [23]:
import random

class Fenwick:
    """Fenwick Tree chỉ số từ 1: cộng vào một vị trí và tính tổng tiền tố, đều O(log m)."""
    def __init__(self, m):
        self.m = m
        self.tree = [0] * (m + 1)          # bỏ trống tree[0]

    def update(self, i, delta=1):          # cnt[i] += delta
        while i <= self.m:
            self.tree[i] += delta
            i += i & -i

    def query(self, i):                    # cnt[1] + ... + cnt[i]
        s = 0
        while i > 0:
            s += self.tree[i]
            i -= i & -i
        return s

def update_path(i, m):
    path = []
    while i <= m:
        path.append(i)
        i += i & -i
    return path

def query_path(i):
    path = []
    while i > 0:
        path.append(i)
        i -= i & -i
    return path

print("update(3) chạm các ô:", update_path(3, 8))
print("query(7)  cộng các ô:", query_path(7))
print("update(5) chạm các ô:", update_path(5, 8), "| query(6) cộng các ô:", query_path(6))

random.seed(1)
m = 20
bit, cnt = Fenwick(m), [0] * (m + 1)
for _ in range(200):
    i = random.randint(1, m)
    bit.update(i)
    cnt[i] += 1
    k = random.randint(0, m)
    assert bit.query(k) == sum(cnt[1:k + 1])
print("Fenwick khớp mảng đếm thường trên 200 thao tác ngẫu nhiên ✅")

update(3) chạm các ô: [3, 4, 8]
query(7)  cộng các ô: [7, 6, 4]
update(5) chạm các ô: [5, 6, 8] | query(6) cộng các ô: [6, 4]
Fenwick khớp mảng đếm thường trên 200 thao tác ngẫu nhiên ✅


## 17. Ghép lại: Cách 5 — Fenwick Tree + nén tọa độ

1. Xây `P`, nén tọa độ: `vals = sorted(set(P))`, tạo `Fenwick(len(vals))` rỗng.
2. Duyệt `p = P[b]` với `b = 0, 1, ..., n` (tính cả `P[0]`):
   - `lo_r = bisect_left(vals, p − upper)` → số giá trị phân biệt `< p − upper`.
   - `hi_r = bisect_right(vals, p − lower)` → số giá trị phân biệt `≤ p − lower`.
   - `count += query(hi_r) − query(lo_r)` → số `P[a]` **đã thêm** nằm trong `[p − upper, p − lower]`.
   - `update(rank(p))` → thêm `p` vào túi **sau khi đếm**.

Vì sao duyệt cả `P[0]`? Ở `b = 0` túi rỗng nên đếm được `0`, sau đó `P[0] = 0` được thêm vào — y hệt việc Cách 3 khởi tạo `seen = [0]`.

**Invariant:** ngay trước khi xử lý `P[b]`, Fenwick chứa đúng các giá trị `P[0], ..., P[b−1]` (mỗi giá trị cộng 1 vào ô hạng của nó). Nên `query(...)` đếm đúng các `a < b`.

In [24]:
def count_range_sum_fenwick(nums: List[int], lower: int, upper: int) -> int:
    P = build_prefix(nums)
    vals = sorted(set(P))                      # nén tọa độ
    bit = Fenwick(len(vals))
    count = 0
    for p in P:
        lo_r = bisect_left(vals, p - upper)    # hạng <= lo_r: giá trị < p - upper
        hi_r = bisect_right(vals, p - lower)   # hạng <= hi_r: giá trị <= p - lower
        count += bit.query(hi_r) - bit.query(lo_r)
        bit.update(bisect_left(vals, p) + 1)   # thêm p (hạng từ 1) SAU khi đếm
    return count

for nums, lower, upper, expected in examples:
    got = count_range_sum_fenwick(nums, lower, upper)
    print(nums, lower, upper, "->", got, "| expected:", expected)
    assert got == expected

[-2, 5, -1] -2 2 -> 3 | expected: 3
[0] 0 0 -> 1 | expected: 1


### Trace cách 5 trên Example 1

`P = [0, -2, 3, 2]`, `vals = [-2, 0, 2, 3]` (hạng `-2→1, 0→2, 2→3, 3→4`), `lower = -2`, `upper = 2`:

| b | `p` | Cửa sổ | `lo_r` | `hi_r` | `query(hi_r) − query(lo_r)` | `count` | Thêm hạng |
|:---:|---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | `[-2, 2]` | 0 | 3 | `0 − 0 = 0` (túi rỗng) | 0 | 2 |
| 1 | -2 | `[-4, 0]` | 0 | 2 | `1 − 0 = 1` (giá trị 0) | 1 | 1 |
| 2 | 3 | `[1, 5]` | 2 | 4 | `2 − 2 = 0` | 1 | 4 |
| 3 | 2 | `[0, 4]` | 1 | 4 | `3 − 1 = 2` (giá trị 0, 3) | 3 | 3 |

Dòng `b = 3`: túi đang có các hạng `{1, 2, 4}` (giá trị `-2, 0, 3`). `query(4) = 3` (cả ba), `query(1) = 1` (chỉ `-2`, nhỏ hơn cận dưới `0`) → hiệu `2`: đó là `0` và `3`, đúng như Cách 3.

In [25]:
def trace_fenwick(nums, lower, upper):
    P = build_prefix(nums)
    vals = sorted(set(P))
    bit = Fenwick(len(vals))
    print("P =", P, "| vals =", vals)
    count = 0
    for b, p in enumerate(P):
        lo_r = bisect_left(vals, p - upper)
        hi_r = bisect_right(vals, p - lower)
        q_hi, q_lo = bit.query(hi_r), bit.query(lo_r)
        count += q_hi - q_lo
        r = bisect_left(vals, p) + 1
        bit.update(r)
        print(f"b={b}, p={p:>2}, cửa sổ=[{p - upper:>2}, {p - lower:>2}], lo_r={lo_r}, hi_r={hi_r}, "
              f"query({hi_r})-query({lo_r}) = {q_hi}-{q_lo} = {q_hi - q_lo}, count={count}, thêm hạng {r}")
    return count

print("Kết quả:", trace_fenwick([-2, 5, -1], -2, 2))

P = [0, -2, 3, 2] | vals = [-2, 0, 2, 3]
b=0, p= 0, cửa sổ=[-2,  2], lo_r=0, hi_r=3, query(3)-query(0) = 0-0 = 0, count=0, thêm hạng 2
b=1, p=-2, cửa sổ=[-4,  0], lo_r=0, hi_r=2, query(2)-query(0) = 1-0 = 1, count=1, thêm hạng 1
b=2, p= 3, cửa sổ=[ 1,  5], lo_r=2, hi_r=4, query(4)-query(2) = 2-2 = 0, count=1, thêm hạng 4
b=3, p= 2, cửa sổ=[ 0,  4], lo_r=1, hi_r=4, query(4)-query(1) = 3-1 = 2, count=3, thêm hạng 3
Kết quả: 3


### Độ phức tạp cách 5

Gọi `N = n + 1` (số prefix sum), `m ≤ N` là số giá trị phân biệt.

- Xây `P`: `O(N)`. Nén tọa độ (`sorted(set(P))`): `O(N log N)`.
- Mỗi `p`: 3 lần `bisect` (`O(log m)`) + 2 lần `query` + 1 lần `update` (`O(log m)` mỗi lần).
- **Thời gian `O(n log n)`**, **bộ nhớ `O(n)`** (`P`, `vals`, `tree`).

So với Cách 4: cùng `O(n log n)`, không đệ quy, code "online" dễ mở rộng (vd: đổi điều kiện cửa sổ). Nhược điểm: phải biết **trước** mọi giá trị để nén (offline) — may mắn là ở bài này ta có đủ `P` từ đầu.

## 18. Code nộp LeetCode — Cách 4 (merge sort đếm khi trộn)

Dưới đây là bản gọn, đúng signature. Ghi chú:

- Trên LeetCode không cần dòng `from typing import List` (đã import sẵn); trong notebook ta đã import ở đầu.
- Bản nộp tự trộn bằng hai con trỏ cho đúng tinh thần merge sort. **Mẹo tăng tốc trong Python:** có thể thay toàn bộ đoạn trộn bằng `P[lo:hi] = sorted(P[lo:hi])`. Thuật toán sắp xếp Timsort của Python tự phát hiện "hai đoạn đã sắp xếp nối nhau" và trộn chúng gần như tuyến tính, lại chạy bằng C nên nhanh hơn vòng lặp Python.

In [26]:
class Solution:
    def countRangeSum(self, nums: List[int], lower: int, upper: int) -> int:
        # Prefix sum: P[0] = 0, P[k+1] = P[k] + nums[k]
        P = [0]
        for x in nums:
            P.append(P[-1] + x)

        # Đếm cặp a < b trong P[lo:hi] với lower <= P[b] - P[a] <= upper, đồng thời sắp xếp P[lo:hi]
        def sort_count(lo: int, hi: int) -> int:
            if hi - lo <= 1:
                return 0
            mid = (lo + hi) // 2
            count = sort_count(lo, mid) + sort_count(mid, hi)

            # Cặp chéo: a ở [lo, mid), b ở [mid, hi) -- hai nửa đã sắp xếp
            i = j = lo
            for b in range(mid, hi):
                while i < mid and P[i] < P[b] - upper:
                    i += 1
                while j < mid and P[j] <= P[b] - lower:
                    j += 1
                count += j - i

            # Trộn hai nửa
            merged = []
            p1, p2 = lo, mid
            while p1 < mid and p2 < hi:
                if P[p1] <= P[p2]:
                    merged.append(P[p1])
                    p1 += 1
                else:
                    merged.append(P[p2])
                    p2 += 1
            merged.extend(P[p1:mid])
            merged.extend(P[p2:hi])
            P[lo:hi] = merged
            return count

        return sort_count(0, len(P))


sol = Solution()
assert sol.countRangeSum([-2, 5, -1], -2, 2) == 3                          # Example 1
assert sol.countRangeSum([0], 0, 0) == 1                                   # Example 2
assert sol.countRangeSum([5], 0, 4) == 0                                   # 1 phần tử, ngoài khoảng
assert sol.countRangeSum([0, 0, 0], 0, 0) == 6                             # mọi đoạn đều tổng 0
assert sol.countRangeSum([1, -1, 1, -1], 0, 0) == 4                        # prefix trùng nhau
assert sol.countRangeSum([1, 2, 3], 3, 5) == 3                             # khoảng không đối xứng
assert sol.countRangeSum([-1, -2], -3, -1) == 3                            # toàn số âm
assert sol.countRangeSum([2147483647, -2147483648, -1, 0], -1, 0) == 4     # giá trị biên 32-bit
print("Solution (merge sort): mọi test PASS ✅")

Solution (merge sort): mọi test PASS ✅


### Bản nộp thay thế — Cách 5 (Fenwick Tree)

Nếu bạn thích lối "duyệt một lượt" không đệ quy, bản Fenwick cũng nộp được. Trong notebook ta đặt tên `SolutionFenwick` để không ghi đè `Solution` ở trên; **khi nộp LeetCode hãy đổi tên thành `Solution`**.

In [27]:
from bisect import bisect_left, bisect_right

class SolutionFenwick:                     # khi nộp LeetCode: đổi tên thành Solution
    def countRangeSum(self, nums: List[int], lower: int, upper: int) -> int:
        P = [0]
        for x in nums:
            P.append(P[-1] + x)
        vals = sorted(set(P))
        m = len(vals)
        tree = [0] * (m + 1)

        def update(i: int) -> None:
            while i <= m:
                tree[i] += 1
                i += i & -i

        def query(i: int) -> int:
            s = 0
            while i > 0:
                s += tree[i]
                i -= i & -i
            return s

        count = 0
        for p in P:
            count += query(bisect_right(vals, p - lower)) - query(bisect_left(vals, p - upper))
            update(bisect_left(vals, p) + 1)
        return count


solf = SolutionFenwick()
assert solf.countRangeSum([-2, 5, -1], -2, 2) == 3
assert solf.countRangeSum([0], 0, 0) == 1
assert solf.countRangeSum([0, 0, 0], 0, 0) == 6
assert solf.countRangeSum([1, -1, 1, -1], 0, 0) == 4
assert solf.countRangeSum([2147483647, -2147483648, -1, 0], -1, 0) == 4
print("SolutionFenwick: mọi test PASS ✅")

SolutionFenwick: mọi test PASS ✅


## 19. Kiểm thử đối chiếu chéo cả 5 cách

Ba lớp kiểm thử:

1. **500 mảng ngẫu nhiên nhỏ** (`n ≤ 12`, giá trị `-5..5`, nhiều số trùng và số âm): mọi cách phải khớp brute force (trọng tài).
2. **Giá trị lớn sát biên 32-bit** (`n = 300`): kiểm tra không có lỗi do số lớn.
3. **`n = 10^5`**: chỉ chạy các cách nhanh, so kết quả với nhau và in thời gian (con số phụ thuộc máy) để thấy `O(n log n)` chạy được cỡ `10^5`, còn `O(n²)` thì không.

Dùng `random.seed(...)` cố định để mỗi lần chạy notebook ra cùng dữ liệu.

In [28]:
import random
import time

methods = {
    "cubic": count_range_sum_cubic,
    "brute O(n²)": count_range_sum_brute,
    "prefix pairs": count_range_sum_prefix_pairs,
    "bisect": count_range_sum_bisect,
    "merge sort": count_range_sum_merge_sort,
    "fenwick": count_range_sum_fenwick,
    "Solution": lambda a, l, u: Solution().countRangeSum(a, l, u),
    "SolutionFenwick": lambda a, l, u: SolutionFenwick().countRangeSum(a, l, u),
}

random.seed(327)
for _ in range(500):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 12))]
    lower = random.randint(-6, 6)
    upper = random.randint(lower, 8)
    expected = count_range_sum_brute(nums, lower, upper)
    for name, f in methods.items():
        got = f(nums[:], lower, upper)
        assert got == expected, (name, nums, lower, upper, got, expected)
print("1) 500 test ngẫu nhiên nhỏ: cả 8 hàm đều khớp brute force ✅")

for _ in range(20):
    nums = [random.choice([2**31 - 1, -(2**31), random.randint(-10**5, 10**5)]) for _ in range(300)]
    lower = random.randint(-10**5, 10**5)
    upper = random.randint(lower, 10**5)
    expected = count_range_sum_brute(nums, lower, upper)
    assert count_range_sum_merge_sort(nums, lower, upper) == expected
    assert count_range_sum_fenwick(nums, lower, upper) == expected
    assert count_range_sum_bisect(nums, lower, upper) == expected
print("2) Giá trị sát biên 32-bit: merge sort, fenwick, bisect khớp brute force ✅")

big = [random.randint(-100, 100) for _ in range(10**5)]
results = {}
for name in ["merge sort", "fenwick", "bisect", "Solution"]:
    start = time.perf_counter()
    results[name] = methods[name](big, -50, 50)
    print(f"3) n = 100000, {name:<10}: đáp án = {results[name]:,}, thời gian ≈ {time.perf_counter() - start:.2f} s")
assert len(set(results.values())) == 1
print("   Các cách nhanh cho cùng đáp án ✅")

1) 500 test ngẫu nhiên nhỏ: cả 8 hàm đều khớp brute force ✅


2) Giá trị sát biên 32-bit: merge sort, fenwick, bisect khớp brute force ✅


3) n = 100000, merge sort: đáp án = 33,389,376, thời gian ≈ 0.38 s


3) n = 100000, fenwick   : đáp án = 33,389,376, thời gian ≈ 0.30 s


3) n = 100000, bisect    : đáp án = 33,389,376, thời gian ≈ 0.88 s


3) n = 100000, Solution  : đáp án = 33,389,376, thời gian ≈ 0.44 s
   Các cách nhanh cho cùng đáp án ✅


## 20. Vì sao đúng? (tổng hợp các invariant bằng lời)

**Nền móng chung — tương ứng 1–1.** Mỗi đoạn `(i, j)` với `i ≤ j` ứng với đúng một cặp `(a, b) = (i, j + 1)` với `a < b` trên mảng `P`, và tổng đoạn bằng `P[b] − P[a]`. Nên "đếm đoạn có tổng trong `[lower, upper]`" **đúng bằng** "đếm cặp `a < b` có `P[b] − P[a]` trong `[lower, upper]`" — không thừa, không thiếu.

**Chuyển vế.** Với `b` cố định: `lower ≤ P[b] − P[a] ≤ upper ⟺ P[b] − upper ≤ P[a] ≤ P[b] − lower`. Đây là biến đổi tương đương (chỉ cộng/trừ hai vế), nên mọi cách dùng cửa sổ đều đếm đúng tập cặp.

**Cách 3 (bisect).** Invariant: *ngay trước khi xử lý `P[b]`, `seen` chứa đúng `P[0..b−1]` và đã sắp xếp.* `bisect_right(seen, R) − bisect_left(seen, L)` đếm đúng số phần tử trong `[L, R]` (kể cả các giá trị trùng). Thêm sau khi đếm ⇒ không bao giờ đếm `a = b`.

**Cách 4 (merge sort).** Mỗi cặp `(a, b)` được đếm đúng một lần tại nút đệ quy đầu tiên tách `a` sang trái, `b` sang phải. Sắp xếp mỗi nửa không đổi **tập** giá trị của nửa đó. Hai ngưỡng tăng dần theo `P[b]` nên two pointers không bỏ sót.

**Cách 5 (Fenwick).** Invariant: *ngay trước khi xử lý `P[b]`, ô hạng `r` trong mảng đếm bằng số lần giá trị `vals[r−1]` xuất hiện trong `P[0..b−1]`.* Nén tọa độ giữ nguyên thứ tự, `bisect` chuyển cửa sổ giá trị thành khoảng hạng chính xác, Fenwick tính tổng tiền tố trên hạng đúng như mảng đếm thường (đã kiểm chứng ngẫu nhiên).

## 21. So sánh các cách

`n = len(nums)`; mọi cách `O(n log n)` đều cần `O(n)` bộ nhớ.

| Cách | Ý tưởng chính | Thời gian | Bộ nhớ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1a. Brute force 3 vòng | Thử mọi đoạn, cộng lại từ đầu | `O(n³)` | `O(1)` | Dịch thẳng từ đề | Rất chậm |
| 1b. Running sum | Cộng dồn khi kéo dài đoạn | `O(n²)` | `O(1)` | Đơn giản, làm trọng tài test | TLE với `n = 10^5` |
| 2. Prefix + đếm cặp | `S = P[b] − P[a]`, thử mọi cặp | `O(n²)` | `O(n)` | Đổi bài toán sang "đếm cặp" | Chưa nhanh hơn |
| 3. `seen` sắp xếp + `bisect` | Cửa sổ `[P−upper, P−lower]`, tìm nhị phân | `O(n²)` worst (do `insort`) | `O(n)` | Ngắn, dễ hiểu, thực tế khá nhanh | Không đảm bảo `O(n log n)` |
| **4. Merge sort đếm khi trộn** | Chia đôi, two pointers đếm cặp chéo | **`O(n log n)`** | `O(n)` | Đảm bảo, chỉ thư viện chuẩn, kỹ thuật kinh điển | Đệ quy, dễ sai thứ tự đếm/trộn |
| 5. Fenwick + nén tọa độ | Mảng đếm theo hạng, tổng tiền tố `O(log)` | **`O(n log n)`** | `O(n)` | Không đệ quy, "online", dễ mở rộng | Phải nén tọa độ, nhiều chi tiết `±1` |

**Gợi ý chọn:** phỏng vấn → trình bày 1b → prefix → cửa sổ → Cách 4 (hoặc 5). Thi đấu/LeetCode bằng Python → Cách 4 với mẹo `sorted`, hoặc Cách 3 nếu chấp nhận rủi ro, hoặc `SortedList` nếu được dùng thư viện ngoài.

## 22. Lỗi người mới hay gặp

| # | Lỗi | Hậu quả | Cách sửa |
|:---:|---|---|---|
| 1 | **Quên `P[0] = 0`** (dùng `accumulate(nums)` không có `initial=0`) | Mất mọi đoạn bắt đầu từ index 0 | Luôn bắt đầu `P = [0]` hoặc `seen = [0]` |
| 2 | **Chuyển vế sai dấu**: dùng cửa sổ `[P + lower, P + upper]` | Thực chất đếm tổng trong `[−upper, −lower]`. Example 1 có `lower = −upper` nên **lỗi bị che**! | Cửa sổ đúng: `[P[b] − upper, P[b] − lower]`; luôn test thêm một khoảng **không đối xứng** |
| 3 | **Lẫn `bisect_left`/`bisect_right`** ở cận trên | Bỏ sót các đoạn có tổng **đúng bằng** `lower` | Cận trên `R` → `bisect_right`; cận dưới `L` → `bisect_left` |
| 4 | **Thêm trước rồi mới đếm** | Đếm thêm cặp `a = b` (đoạn rỗng) mỗi khi `0 ∈ [lower, upper]` | Đếm trước, thêm sau (như Two Sum) |
| 5 | **Sắp xếp toàn bộ `P` rồi đếm cặp** (hoặc trong merge sort: trộn xong mới đếm) | Mất thông tin `a < b`, đếm "hiệu dương" thay vì `P[b] − P[a]` | Chỉ sắp xếp **bên trong từng nửa**; đếm cặp chéo **trước** khi trộn |
| 6 | **Dùng sliding window** như bài toàn số không âm | Sai khi có số âm vì tổng không đơn điệu | Dùng prefix + cửa sổ (hoặc chỉ dùng sliding window khi chắc chắn `nums[i] ≥ 0`) |
| 7 | Viết C++/Java dùng `int` cho prefix sum | Tràn số (`P` tới khoảng `2·10^14`) | Dùng `long long` / `long`; Python không bị |

Cell dưới cài đúng 6 phiên bản sai (lỗi 1–6) và chạy trên 3 bộ test: Example 1 (khoảng đối xứng), `[1, 2, 3]` với `[3, 5]` (không đối xứng), `[-1, -2]` với `[-3, -1]` (toàn số âm).

In [29]:
def wrong_no_p0(nums, lower, upper):               # Lỗi 1: quên P[0] = 0
    P = list(accumulate(nums))                      # thiếu số 0 ở đầu
    return sum(lower <= P[b] - P[a] <= upper for b in range(len(P)) for a in range(b))

def wrong_window_sign(nums, lower, upper):          # Lỗi 2: cửa sổ [P + lower, P + upper]
    seen, prefix, count = [0], 0, 0
    for x in nums:
        prefix += x
        count += bisect_right(seen, prefix + upper) - bisect_left(seen, prefix + lower)
        insort(seen, prefix)
    return count

def wrong_bisect_side(nums, lower, upper):          # Lỗi 3: bisect_left cho cận trên
    seen, prefix, count = [0], 0, 0
    for x in nums:
        prefix += x
        count += bisect_left(seen, prefix - lower) - bisect_left(seen, prefix - upper)
        insort(seen, prefix)
    return count

def wrong_insert_first(nums, lower, upper):         # Lỗi 4: thêm trước rồi mới đếm
    seen, prefix, count = [0], 0, 0
    for x in nums:
        prefix += x
        insort(seen, prefix)
        count += bisect_right(seen, prefix - lower) - bisect_left(seen, prefix - upper)
    return count

def wrong_sort_all(nums, lower, upper):             # Lỗi 5: sắp xếp toàn bộ P rồi mới đếm
    P = sorted(build_prefix(nums))
    return sum(lower <= P[b] - P[a] <= upper for b in range(len(P)) for a in range(b))

def count_at_most_window(nums, k):                  # đếm đoạn có tổng <= k: CHỈ đúng khi nums >= 0
    left = total = count = 0
    for right, x in enumerate(nums):
        total += x
        while left <= right and total > k:
            total -= nums[left]
            left += 1
        count += right - left + 1
    return count

def wrong_sliding_window(nums, lower, upper):       # Lỗi 6: sliding window khi có số âm
    return count_at_most_window(nums, upper) - count_at_most_window(nums, lower - 1)

tests = [([-2, 5, -1], -2, 2), ([1, 2, 3], 3, 5), ([-1, -2], -3, -1)]
print(f"{'hàm':<22}" + "".join(f"{str(t[0]) + ' ' + str([t[1], t[2]]):>22}" for t in tests))
print(f"{'ĐÚNG (merge sort)':<22}" + "".join(f"{count_range_sum_merge_sort(*t):>22}" for t in tests))
for f in [wrong_no_p0, wrong_window_sign, wrong_bisect_side, wrong_insert_first, wrong_sort_all, wrong_sliding_window]:
    print(f"{f.__name__:<22}" + "".join(f"{f(*t):>22}" for t in tests))

hàm                      [-2, 5, -1] [-2, 2]      [1, 2, 3] [3, 5]     [-1, -2] [-3, -1]
ĐÚNG (merge sort)                          3                     3                     3
wrong_no_p0                                1                     2                     1
wrong_window_sign                          3                     0                     0
wrong_bisect_side                          2                     1                     2
wrong_insert_first                         6                     3                     3
wrong_sort_all                             3                     3                     0
wrong_sliding_window                       2                     3                     3


### Đọc kết quả bảng lỗi

- **`wrong_no_p0`** ra `1` ở Example 1: mất hai đoạn `[0..0]` và `[0..2]` (đều bắt đầu từ index 0).
- **`wrong_window_sign`** ra **đúng `3`** ở Example 1 — vì khoảng `[-2, 2]` đối xứng nên đếm `[−upper, −lower]` cũng là `[-2, 2]`. Chỉ khi thử `[1, 2, 3]` với `[3, 5]` mới lộ ra `0`. Bài học: **đừng chỉ test bằng ví dụ của đề**.
- **`wrong_bisect_side`** ra `2` ở Example 1: bỏ sót đoạn `[0..0]` có tổng `-2` đúng bằng `lower` (giá trị `P[a] = 0` nằm đúng ở cận trên cửa sổ `[-4, 0]`).
- **`wrong_insert_first`** ra `6 = 3 + 3`: mỗi `b` đếm thừa một "đoạn rỗng" vì `0 ∈ [-2, 2]`. Ở hai test còn lại (`[3, 5]` và `[-3, -1]`), `0` không nằm trong khoảng nên lỗi bị che và kết quả trùng hợp đúng.
- **`wrong_sort_all`** đúng ở hai test đầu một cách tình cờ, nhưng ra `0` ở test toàn số âm: sau khi sắp xếp, mọi hiệu "phải trừ trái" đều `≥ 0`, không thể rơi vào `[-3, -1]`.
- **`wrong_sliding_window`** ra `2` ở Example 1 vì có số âm làm tổng không đơn điệu. Nó **đúng** ở `[1, 2, 3]` (toàn số dương) — đúng như lý thuyết: sliding window chỉ an toàn khi mọi phần tử không âm. Ở test `[-1, -2]` nó cũng ra `3` nhưng chỉ là **tình cờ** — chạy đúng trên một vài ví dụ không chứng minh được thuật toán đúng.

## 23. Mẹo nhận dạng pattern và các bài liên quan

**Dấu hiệu nhận dạng:**

- Đề hỏi **"đếm số đoạn con (subarray) có tổng thỏa điều kiện"** → nghĩ ngay **prefix sum**: tổng đoạn = `P[b] − P[a]`, bài toán thành **đếm cặp `a < b`**.
- Điều kiện là **"bằng `k`"** → `dict`/`Counter` đếm tần suất prefix: `O(n)` (LeetCode 560).
- Điều kiện là **"nằm trong khoảng"** hoặc **"lớn hơn / nhỏ hơn"**, và mảng **có số âm** → cần cấu trúc đếm theo **thứ tự**: merge sort đếm khi trộn, Fenwick/Segment Tree + nén tọa độ, hoặc list/cây sắp xếp + binary search.
- Mảng **toàn số không âm** → tổng đơn điệu → **sliding window** `O(n)` là đủ (dùng `atMost(upper) − atMost(lower − 1)`).
- Thấy **"đếm cặp `i < j` với điều kiện so sánh giữa `f(i)` và `g(j)`"** → khuôn "đếm nghịch thế" bằng merge sort hoặc BIT.

**Bài liên quan (cùng khuôn):**

| Bài | Độ khó | Điểm chung với 327 |
|---|---|---|
| 560. Subarray Sum Equals K | Medium | Prefix sum + `dict` (điều kiện "bằng") |
| 974. Subarray Sums Divisible by K | Medium | Prefix sum theo modulo + đếm tần suất |
| 315. Count of Smaller Numbers After Self | Hard | Đếm cặp `i < j` với `nums[j] < nums[i]`: BIT + nén tọa độ / merge sort |
| 493. Reverse Pairs | Hard | Đếm cặp `i < j` với `nums[i] > 2·nums[j]`: merge sort đếm trước khi trộn |
| 2426. Number of Pairs Satisfying Inequality | Hard | Biến đổi về đếm cặp `i < j` với `d[i] ≤ d[j] + diff`: merge sort / BIT |
| 862. Shortest Subarray with Sum at Least K | Hard | Prefix sum có số âm, nhưng dùng monotonic deque |

Cell dưới minh họa hai "người anh em" dễ: (1) LeetCode 560 chính là 327 với `lower = upper = k`, giải bằng `defaultdict` — một `dict` tự tạo giá trị mặc định `0` khi truy cập key chưa có; (2) khi `nums ≥ 0`, sliding window ở trên cho kết quả đúng trong `O(n)`.

In [30]:
import random
from collections import defaultdict

def subarray_sum_equals_k(nums, k):          # LeetCode 560
    freq = defaultdict(int)                  # key chưa có -> mặc định 0
    freq[0] = 1                              # P[0] = 0
    prefix = count = 0
    for x in nums:
        prefix += x
        count += freq[prefix - k]            # số P[a] == P[b] - k đã gặp
        freq[prefix] += 1                    # thêm sau khi đếm
    return count

print("560 Example:", subarray_sum_equals_k([1, 1, 1], 2), subarray_sum_equals_k([1, 2, 3], 3))   # 2 2

random.seed(560)
for _ in range(300):
    nums = [random.randint(-4, 4) for _ in range(random.randint(1, 10))]
    k = random.randint(-5, 5)
    assert subarray_sum_equals_k(nums, k) == count_range_sum_brute(nums, k, k)
print("560 == 327 với lower = upper = k ✅")

for _ in range(300):
    nums = [random.randint(0, 5) for _ in range(random.randint(1, 10))]     # toàn số không âm
    lower = random.randint(0, 8)
    upper = random.randint(lower, 12)
    assert wrong_sliding_window(nums, lower, upper) == count_range_sum_brute(nums, lower, upper)
print("Khi nums >= 0: sliding window O(n) khớp brute force ✅")

560 Example: 2 2
560 == 327 với lower = upper = k ✅
Khi nums >= 0: sliding window O(n) khớp brute force ✅


## 24. Bài tập vận dụng

Sáu bài từ dễ đến khó. Hãy tự làm trước, xem **Gợi ý theo mức** (mục 25) khi bí, cuối cùng mới xem **Đáp án** (mục 26).

### Bài 1 — Đoán output (không chạy code)

```python
nums = [3, -1, 2]
print(build_prefix(nums))                       # (a) ?

a = [-1, 0, 0, 2, 2, 5]
print(bisect_left(a, 2), bisect_right(a, 2))    # (b) ?
print(bisect_right(a, 4) - bisect_left(a, 0))   # (c) ? (đếm số phần tử trong [0, 4])
```

### Bài 2 — Tính bằng tay

**(a)** Với `nums = [3, -1, 2]`, `lower = 1`, `upper = 3`: lập mảng `P`; với mỗi `b = 1, 2, 3` tính cửa sổ `[P[b] − upper, P[b] − lower]`, liệt kê các `P[a]` (`a < b`) rơi vào cửa sổ, rồi cộng lại ra đáp án. Đối chiếu bằng cách liệt kê cả 6 đoạn.

**(b)** Ở một nút merge sort: `left = [-3, 0, 1]`, `right = [-1, 2, 4]` (đã sắp xếp), `lower = 1`, `upper = 3`. Lập bảng two pointers như mục 12 (cột `y`, cửa sổ, `lo`, `hi`, `left[lo:hi]`) và cho biết số cặp chéo.

### Bài 3 — Điền TODO: two pointers đếm cặp chéo

Hoàn thiện hàm đếm số cặp `(x ∈ left, y ∈ right)` với `lower ≤ y − x ≤ upper`, biết hai list đã sắp xếp tăng dần. Yêu cầu `O(len(left) + len(right))` — **không** dùng hai vòng lồng nhau thử mọi cặp.

In [31]:
def exercise_count_cross_pairs(left: List[int], right: List[int], lower: int, upper: int) -> int:
    lo = hi = 0
    count = 0
    for y in right:
        # TODO 1: tiến lo khi left[lo] < y - upper
        # TODO 2: tiến hi khi left[hi] <= y - lower
        # TODO 3: cộng số phần tử trong left[lo:hi] vào count
        pass
    return count

# Tự làm xong rồi mới mở test:
# assert exercise_count_cross_pairs([-2, 0, 4], [0, 1, 2], -1, 2) == 4
# assert exercise_count_cross_pairs([-3, 0, 1], [-1, 2, 4], 1, 3) == 4
# assert exercise_count_cross_pairs([], [1, 2], 0, 5) == 0

### Bài 4 — Tự viết lại Cách 4 từ đầu

Không nhìn code phía trên, tự viết hàm `exercise_count_range_sum` theo merge sort: xây prefix (nhớ `P[0] = 0`), hàm đệ quy `sort_count(lo, hi)` (base case, đệ quy hai nửa, đếm cặp chéo **trước**, trộn **sau**).

In [32]:
def exercise_count_range_sum(nums: List[int], lower: int, upper: int) -> int:
    # TODO 1: xây P với P[0] = 0
    # TODO 2: viết sort_count(lo, hi): base case hi - lo <= 1
    # TODO 3: đệ quy hai nửa, đếm cặp chéo bằng two pointers
    # TODO 4: trộn hai nửa vào P[lo:hi]
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_count_range_sum([-2, 5, -1], -2, 2) == 3
# assert exercise_count_range_sum([0], 0, 0) == 1
# assert exercise_count_range_sum([1, 2, 3], 3, 5) == 3
# assert exercise_count_range_sum([-1, -2], -3, -1) == 3

### Bài 5 — Biến thể: LeetCode 315 — Count of Smaller Numbers After Self

Cho `nums`, trả về list `counts` với `counts[i]` = số phần tử **bên phải** `nums[i]` mà **nhỏ hơn** `nums[i]`. (Constraints của đề: `1 <= nums.length <= 10^5`, `-10^4 <= nums[i] <= 10^4`.)

Ví dụ: `nums = [5, 2, 6, 1]` → `[2, 1, 1, 0]` (bên phải 5 có 2 và 1 nhỏ hơn; bên phải 2 có 1; bên phải 6 có 1; bên phải 1 không có gì).

Yêu cầu: dùng **Fenwick Tree + nén tọa độ**, `O(n log n)`. Hướng: duyệt **từ phải sang trái**, "túi" chứa các số đã gặp (ở bên phải).

In [33]:
def exercise_count_smaller(nums: List[int]) -> List[int]:
    # TODO 1: nén tọa độ vals = sorted(set(nums)), tạo Fenwick(len(vals))
    # TODO 2: duyệt i từ n - 1 về 0
    # TODO 3: counts[i] = số giá trị đã thêm có hạng nhỏ hơn hạng của nums[i]
    # TODO 4: thêm nums[i] vào Fenwick SAU khi đếm
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]
# assert exercise_count_smaller([-1]) == [0]
# assert exercise_count_smaller([-1, -1]) == [0, 0]

### Bài 6 — Mini challenge: LeetCode 493 — Reverse Pairs

Cho `nums`, đếm số cặp `(i, j)` với `0 <= i < j < n` và **`nums[i] > 2 * nums[j]`**. (Constraints của đề: `1 <= nums.length <= 5·10^4`, `-2^31 <= nums[i] <= 2^31 - 1`.)

Ví dụ: `[1, 3, 2, 3, 1]` → `2` (cặp `(1, 4)`: `3 > 2·1` và `(3, 4)`: `3 > 2·1`); `[2, 4, 3, 5, 1]` → `3`.

Yêu cầu: dùng **merge sort đếm cặp chéo trước khi trộn**, `O(n log n)`. Đây là bài "anh em ruột" của 327: thay điều kiện cửa sổ bằng điều kiện `x > 2y`.

In [34]:
def exercise_reverse_pairs(nums: List[int]) -> int:
    # TODO 1: copy nums vào list a (để không sửa input)
    # TODO 2: sort_count(lo, hi) như Cách 4
    # TODO 3: đếm cặp chéo: với mỗi x ở nửa trái (đã sắp), đếm số y ở nửa phải (đã sắp) có x > 2 * y
    #         gợi ý: con trỏ trên nửa phải chỉ tiến khi x tăng
    # TODO 4: trộn hai nửa
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert exercise_reverse_pairs([1, 3, 2, 3, 1]) == 2
# assert exercise_reverse_pairs([2, 4, 3, 5, 1]) == 3
# assert exercise_reverse_pairs([2147483647, 2147483647, 2147483647]) == 0

## 25. Gợi ý theo mức

**Bài 1**
- Mức 1: `P` dài hơn `nums` một ô và bắt đầu bằng `0`.
- Mức 2: `bisect_left` = số phần tử `< x`; `bisect_right` = số phần tử `<= x`.
- Mức 3: (c) = (số phần tử `<= 4`) − (số phần tử `< 0`).

**Bài 2**
- Mức 1: (a) `P = [0, 3, 2, 4]`; cửa sổ của `b` là `[P[b] − 3, P[b] − 1]`.
- Mức 2: (b) cửa sổ của `y` là `[y − 3, y − 1]`; `lo` dừng ở số đầu tiên `>= y − 3`, `hi` dừng ở số đầu tiên `> y − 1`.
- Mức 3: (b) với `y = -1` cửa sổ là `[-4, -2]` → chỉ có `-3`.

**Bài 3**
- Mức 1: hai con trỏ chỉ tiến vì `y` tăng dần.
- Mức 2: `while lo < len(left) and left[lo] < y - upper: lo += 1` — tương tự cho `hi` với `<= y - lower`.
- Mức 3: sau hai vòng `while`, `count += hi - lo`.

**Bài 4**
- Mức 1: chép lại khung 5 bước ở mục 13.
- Mức 2: cặp chéo duyệt `b` trong `range(mid, hi)`, con trỏ `i`, `j` bắt đầu từ `lo` và không vượt `mid`.
- Mức 3: trộn bằng `merged` + hai con trỏ `p1 = lo`, `p2 = mid`, cuối cùng `P[lo:hi] = merged` (hoặc tạm dùng `sorted(P[lo:hi])`).

**Bài 5**
- Mức 1: duyệt từ phải sang trái thì "bên phải `i`" chính là "những số đã gặp".
- Mức 2: hạng `r = bisect_left(vals, nums[i]) + 1`; số đã thêm **nhỏ hơn hẳn** `nums[i]` là `query(r − 1)`.
- Mức 3: `counts[i] = bit.query(r - 1)` rồi `bit.update(r)`.

**Bài 6**
- Mức 1: giống Cách 4 nhưng điều kiện cặp chéo là `left_value > 2 * right_value`.
- Mức 2: duyệt `i` trên nửa trái (tăng dần), con trỏ `j` trên nửa phải tiến khi `a[i] > 2 * a[j]`; số cặp của `a[i]` là `j − mid`.
- Mức 3: vì `a[i]` tăng dần nên `j` không bao giờ phải lùi; nhớ đếm **trước** khi trộn.

## 26. Đáp án tham khảo

**Bài 1.** (a) `[0, 3, 2, 4]`. (b) `3 5` — có 3 số `< 2` (`-1, 0, 0`) và 5 số `<= 2`. (c) `bisect_right(a, 4) − bisect_left(a, 0) = 5 − 1 = 4` (các số `0, 0, 2, 2`).

**Bài 2 (a).** `P = [0, 3, 2, 4]`, cửa sổ `[P[b] − 3, P[b] − 1]`:

| b | `P[b]` | Cửa sổ | `P[a]` đã gặp | Rơi vào cửa sổ | + |
|:---:|---:|:---:|:---|:---|:---:|
| 1 | 3 | `[0, 2]` | `0` | `0` | 1 |
| 2 | 2 | `[-1, 1]` | `0, 3` | `0` | 1 |
| 3 | 4 | `[1, 3]` | `0, 3, 2` | `3, 2` | 2 |

Đáp án **4**. Đối chiếu 6 đoạn: `[3]=3` ✅, `[3,-1]=2` ✅, `[3,-1,2]=4` ❌, `[-1]=-1` ❌, `[-1,2]=1` ✅, `[2]=2` ✅ → 4 ✅.

**Bài 2 (b).** `left = [-3, 0, 1]`, cửa sổ `[y − 3, y − 1]`:

| y | Cửa sổ | `lo` | `hi` | `left[lo:hi]` | + |
|:---:|:---:|:---:|:---:|:---|:---:|
| -1 | `[-4, -2]` | 0 | 1 | `[-3]` | 1 |
| 2 | `[-1, 1]` | 1 | 3 | `[0, 1]` | 2 |
| 4 | `[1, 3]` | 2 | 3 | `[1]` | 1 |

Đáp án **4** cặp chéo. Cell dưới kiểm chứng Bài 1 và Bài 2 bằng code; cell tiếp theo là đáp án Bài 3–6 kèm test.

In [35]:
# Kiểm chứng Bài 1
nums = [3, -1, 2]
a = [-1, 0, 0, 2, 2, 5]
assert build_prefix(nums) == [0, 3, 2, 4]
assert (bisect_left(a, 2), bisect_right(a, 2)) == (3, 5)
assert bisect_right(a, 4) - bisect_left(a, 0) == 4
print("Bài 1:", build_prefix(nums), bisect_left(a, 2), bisect_right(a, 2), bisect_right(a, 4) - bisect_left(a, 0))

# Kiểm chứng Bài 2
assert count_range_sum_brute([3, -1, 2], 1, 3) == 4
print("Bài 2(a): đáp án =", count_range_sum_brute([3, -1, 2], 1, 3))
print("Bài 2(b):", count_cross_pairs([-3, 0, 1], [-1, 2, 4], 1, 3, verbose=True))

Bài 1: [0, 3, 2, 4] 3 5 4
Bài 2(a): đáp án = 4
y=-1: cửa sổ=[-4, -2], lo=0, hi=1, left[lo:hi]=[-3] -> +1
y=2: cửa sổ=[-1,  1], lo=1, hi=3, left[lo:hi]=[0, 1] -> +2
y=4: cửa sổ=[ 1,  3], lo=2, hi=3, left[lo:hi]=[1] -> +1
Bài 2(b): 4


In [36]:
import random

# ---------- Bài 3
def answer_count_cross_pairs(left: List[int], right: List[int], lower: int, upper: int) -> int:
    lo = hi = 0
    count = 0
    for y in right:
        while lo < len(left) and left[lo] < y - upper:
            lo += 1
        while hi < len(left) and left[hi] <= y - lower:
            hi += 1
        count += hi - lo
    return count

assert answer_count_cross_pairs([-2, 0, 4], [0, 1, 2], -1, 2) == 4
assert answer_count_cross_pairs([-3, 0, 1], [-1, 2, 4], 1, 3) == 4
assert answer_count_cross_pairs([], [1, 2], 0, 5) == 0

# ---------- Bài 4
def answer_count_range_sum(nums: List[int], lower: int, upper: int) -> int:
    P = [0]
    for x in nums:
        P.append(P[-1] + x)

    def sort_count(lo: int, hi: int) -> int:
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        count = sort_count(lo, mid) + sort_count(mid, hi)
        count += answer_count_cross_pairs(P[lo:mid], P[mid:hi], lower, upper)   # đếm TRƯỚC
        P[lo:hi] = merge_two_sorted(P[lo:mid], P[mid:hi])                       # trộn SAU
        return count

    return sort_count(0, len(P))

assert answer_count_range_sum([-2, 5, -1], -2, 2) == 3
assert answer_count_range_sum([0], 0, 0) == 1
assert answer_count_range_sum([1, 2, 3], 3, 5) == 3
assert answer_count_range_sum([-1, -2], -3, -1) == 3

# ---------- Bài 5 (LeetCode 315)
def answer_count_smaller(nums: List[int]) -> List[int]:
    vals = sorted(set(nums))
    bit = Fenwick(len(vals))
    counts = [0] * len(nums)
    for i in range(len(nums) - 1, -1, -1):        # từ phải sang trái
        r = bisect_left(vals, nums[i]) + 1
        counts[i] = bit.query(r - 1)              # số đã gặp có hạng < r, tức nhỏ hơn hẳn nums[i]
        bit.update(r)
    return counts

assert answer_count_smaller([5, 2, 6, 1]) == [2, 1, 1, 0]
assert answer_count_smaller([-1]) == [0]
assert answer_count_smaller([-1, -1]) == [0, 0]

# ---------- Bài 6 (LeetCode 493)
def answer_reverse_pairs(nums: List[int]) -> int:
    a = nums[:]

    def sort_count(lo: int, hi: int) -> int:
        if hi - lo <= 1:
            return 0
        mid = (lo + hi) // 2
        count = sort_count(lo, mid) + sort_count(mid, hi)
        j = mid
        for i in range(lo, mid):                   # a[i] tăng dần
            while j < hi and a[i] > 2 * a[j]:
                j += 1
            count += j - mid                       # a[mid..j-1] đều thỏa a[i] > 2 * a[j]
        a[lo:hi] = merge_two_sorted(a[lo:mid], a[mid:hi])
        return count

    return sort_count(0, len(a))

assert answer_reverse_pairs([1, 3, 2, 3, 1]) == 2
assert answer_reverse_pairs([2, 4, 3, 5, 1]) == 3
assert answer_reverse_pairs([2147483647, 2147483647, 2147483647]) == 0

# ---------- Đối chiếu ngẫu nhiên với brute force
random.seed(2026)
for _ in range(300):
    nums = [random.randint(-6, 6) for _ in range(random.randint(1, 10))]
    lower = random.randint(-6, 6)
    upper = random.randint(lower, 8)
    assert answer_count_range_sum(nums, lower, upper) == count_range_sum_brute(nums, lower, upper)
    assert answer_count_smaller(nums) == [sum(nums[j] < nums[i] for j in range(i + 1, len(nums))) for i in range(len(nums))]
    assert answer_reverse_pairs(nums) == sum(nums[i] > 2 * nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums)))
print("Đáp án Bài 3, 4, 5, 6: mọi test PASS (kể cả 300 test ngẫu nhiên so với brute force) ✅")

Đáp án Bài 3, 4, 5, 6: mọi test PASS (kể cả 300 test ngẫu nhiên so với brute force) ✅


## 27. Template ngắn nên nhớ

```python
class Solution:
    def countRangeSum(self, nums, lower, upper):
        P = [0]                                   # 1) prefix, nhớ P[0] = 0
        for x in nums:
            P.append(P[-1] + x)

        def sort_count(lo, hi):                   # 2) đếm cặp a < b trong P[lo:hi] + sắp xếp nó
            if hi - lo <= 1:
                return 0
            mid = (lo + hi) // 2
            cnt = sort_count(lo, mid) + sort_count(mid, hi)
            i = j = lo
            for b in range(mid, hi):              # 3) cặp chéo: cửa sổ [P[b]-upper, P[b]-lower]
                while i < mid and P[i] < P[b] - upper:
                    i += 1
                while j < mid and P[j] <= P[b] - lower:
                    j += 1
                cnt += j - i
            P[lo:hi] = sorted(P[lo:hi])           # 4) trộn SAU khi đếm (hoặc tự merge)
            return cnt

        return sort_count(0, len(P))
```

**Câu thần chú:** *"Tổng đoạn = hiệu hai prefix → đếm cặp `a < b` → chuyển vế ra cửa sổ `[P[b] − upper, P[b] − lower]` → đếm 'số đã gặp trong cửa sổ' bằng merge sort / Fenwick / bisect."*

## 28. Checklist trước khi submit

- Đã có **`P[0] = 0`** (hoặc `seen = [0]`) để đếm các đoạn bắt đầu từ index 0 chưa?
- Cửa sổ có đúng **`[P[b] − upper, P[b] − lower]`** (chéo dấu) chưa? Đã test thêm một khoảng **không đối xứng** chưa?
- Cận trên dùng **`bisect_right` / `<=`**, cận dưới dùng **`bisect_left` / `<`** chưa?
- **Đếm trước, thêm/trộn sau** chưa (không đếm đoạn rỗng `a = b`, không mất thông tin trái/phải)?
- Merge sort: base case `hi − lo <= 1`, hai con trỏ chỉ tiến và **không vượt `mid`** chưa?
- Fenwick: hạng bắt đầu từ **1**, `query(0) = 0`, nén tọa độ bằng `sorted(set(P))` chưa?
- Đã thử: hai example của đề, mảng 1 phần tử, toàn số 0, toàn số âm, prefix trùng nhau, giá trị sát `±2^31` chưa?
- Viết ngôn ngữ khác Python: prefix sum đã dùng kiểu **64-bit** chưa?